# C3: первый запуск на Kaggle

**Этот notebook подготовлен, но ни одна его ячейка не выполнялась автором.**

Что он делает: загружает **train + validation** из ogbl-biokg, проверяет данные и
сравнивает три простые стратегии. По отдельному выбору запускает **одну** маленькую
обучаемую модель. Test не используется. Это первый пилот, а не полный проект.

Сначала достаточно режима `baselines` на **CPU**. Включать T4 нужно для `pilot`.
Если Kaggle выдаёт две T4, код использует только `cuda:0`.

По умолчанию `MODE = "prepare_only"`: даже Run All не начнёт загрузку или эксперимент.
Данные, веса и таблицы сохраняются в `/kaggle/working/dkr_c3`. Ничего не очищается.


## 1. Выбери, что сделать

Для **первого baseline** поставь:
```python
MODE = "baselines"
ALLOW_DOWNLOAD = True
ALLOW_OGB_INSTALL = True
```

Понадобится включённый Internet. Разрешение установки касается только маленького
пакета OGB без зависимостей; готовый Torch Kaggle не переустанавливается.
Датасет большой: архив, извлечённые файлы и кеш остаются на диске.

Если публичные файлы OGB уже добавлены как Kaggle Input, укажи их папку в `INPUT_DIR`.
В ней ожидаются `split/random/train.pt` и `valid.pt` (или подпапка `biokg`).
Используй только доверенные оригинальные OGB `.pt`: этот формат содержит pickle.


In [5]:
from pathlib import Path

MODE = "baselines"           # "prepare_only", "baselines", "pilot"
ALLOW_DOWNLOAD = True           # Разрешить загрузку публичного OGB
ALLOW_OGB_INSTALL = True         # Установить только ogb==1.3.6, без deps
INPUT_DIR = None                 # Например Path("/kaggle/input/my-public-biokg/biokg")

# Одна и та же небольшая validation-выборка для всех моделей.
MAX_VALID_PER_RELATION = 20       # None = вся validation, заметно дольше
VALIDATION_SEED = 71
EVAL_BATCH = 64
CANDIDATE_CHUNK = 128

# В режиме pilot обучается только эта модель, не три сразу.
PILOT_MODEL = "head_blind"        # "head_blind", "complex", "transe"
PILOT_DIM = 128                  # Real scalars/entity; ComplEx rank=64
PILOT_STEPS = 500
PILOT_BATCH = 256
PILOT_NEGATIVES = 64
PILOT_SEED = 101
PILOT_LR = 0.001
PILOT_MAX_MINUTES = 20

WORKSPACE = Path("/kaggle/working/dkr_c3")
if MODE not in {"prepare_only", "baselines", "pilot"}:
    raise ValueError("Неверный MODE")
if MODE != "prepare_only" and not Path("/kaggle/working").is_dir():
    raise RuntimeError("Этот notebook запускается в Kaggle. Локальная установка не используется.")
print("Режим:", MODE)
if MODE == "prepare_only":
    print("Загрузка, baseline, проверки и обучение отключены. Измени MODE только для своего запуска.")


Режим: baselines


## 2. Подготовка кода

Внутри notebook уже есть исходники проекта. Ничего отдельно скачивать с GitHub
не нужно. Большая ячейка ниже только сохраняет эти исходники в рабочую папку
Kaggle; она не обучает модель. Если такой же файл уже есть, он используется повторно.
Если содержимое отличается, код остановится, сохранив старый файл.


In [6]:
SOURCE_BUNDLE = {'__init__.py': '"""C3 pilot tools. Importing this package does not load data or run models."""\n\n__version__ = \'0.1.0\'\n', 'data.py': '"""Safe, validation-only BioKG preparation for a Kaggle/Colab workspace.\n\nInputs are the public OGB train.pt/valid.pt files. Outputs use typed GLOBAL\nentity IDs and read-only .npy memory maps. No test file is opened, extracted,\nor passed to a trainer. Calling load_data is an action; importing is inert.\n"""\n\nfrom __future__ import annotations\n\nimport csv\nimport gzip\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport shutil\nimport stat\nfrom datetime import datetime, timezone\nimport urllib.request\nimport uuid\nimport zipfile\n\nimport numpy as np\n\n\nTYPE_NAMES = ("disease", "drug", "function", "protein", "sideeffect")\nTYPE_SIZES = np.array([10687, 10533, 45085, 17499, 9969], dtype=np.int64)\nTYPE_OFFSETS = np.array([0, 10687, 21220, 66305, 83804], dtype=np.int64)\nN_ENTITIES = 93773\nN_RELATIONS = 51\nEXPECTED_COUNTS = {"train": 4762678, "valid": 162886}\nOFFICIAL_URL = "https://snap.stanford.edu/ogb/data/linkproppred/biokg.zip"\nCACHE_VERSION = 1\nCHUNK_ROWS = 1024\n_SPLIT_COLUMNS = ("row_id", "head", "relation", "tail", "head_type", "tail_type")\n_SOURCE_REFS = {\n    "loader": "https://github.com/snap-stanford/ogb/blob/master/ogb/linkproppred/dataset.py",\n    "typed_ids": "https://github.com/snap-stanford/ogb/blob/master/examples/linkproppred/biokg/dataloader.py",\n    "node_metadata": "https://github.com/snap-stanford/ogb/blob/master/ogb/io/read_graph_raw.py",\n}\n\n\ndef _utc() -> str:\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef _sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef _version(package: str) -> str | None:\n    try:\n        return importlib.metadata.version(package)\n    except importlib.metadata.PackageNotFoundError:\n        return None\n\n\ndef _no_links(path: Path) -> Path:\n    """Reject existing symlink/junction components, including input parents."""\n    path = path.expanduser().absolute()\n    for item in (path, *path.parents):\n        if item.is_symlink() or (hasattr(item, "is_junction") and item.is_junction()):\n            raise ValueError(f"Symlinks/junctions are not allowed: {item}")\n    return path\n\n\ndef _write_json_new(path: Path, payload: dict) -> None:\n    with path.open("x", encoding="utf-8") as stream:\n        json.dump(payload, stream, ensure_ascii=False, indent=2, allow_nan=False)\n        stream.write("\\n")\n\n\ndef _mapping() -> dict:\n    return {\n        "version": "biokg_global_lexicographic_v1",\n        "type_order": list(TYPE_NAMES),\n        "type_sizes": TYPE_SIZES.tolist(),\n        "type_offsets": TYPE_OFFSETS.tolist(),\n        "global_id_formula": "type_offsets[type_id] + local_id",\n        "n_entities": N_ENTITIES,\n        "n_relations": N_RELATIONS,\n        "source": "docs/SPEC.md section 4; checked against raw num-node-dict when present",\n    }\n\n\ndef _find_public_root(input_dir: Path) -> Path:\n    """Only exact supported locations; no recursive walks or test reads."""\n    root = _no_links(input_dir)\n    candidates = (root, root / "biokg", root / "ogbl_biokg")\n    matches = []\n    for candidate in candidates:\n        _no_links(candidate)\n        if all(_no_links(candidate / "split" / "random" / f"{s}.pt").is_file()\n               for s in ("train", "valid")):\n            matches.append(candidate)\n    if len(matches) != 1:\n        raise FileNotFoundError(\n            "INPUT_DIR must uniquely contain split/random/train.pt and valid.pt "\n            "directly, under biokg/, or under ogbl_biokg/. No recursive search."\n        )\n    return matches[0]\n\n\ndef _download_public(data_dir: Path) -> Path:\n    """Explicit opt-in, append-only download and selective extraction.\n\n    A unique exclusive directory contains both names in the partial->complete\n    rename. Interrupted partial files and the complete ZIP are retained.\n    """\n    target_dir = _no_links(data_dir / f"public_ogb_{uuid.uuid4().hex}")\n    target_dir.mkdir(mode=0o700, parents=False, exist_ok=False)\n    partial = target_dir / "biokg.zip.partial"\n    archive = target_dir / "biokg.zip"\n    print("Downloading public OGB archive; it will be retained:", OFFICIAL_URL, flush=True)\n    with urllib.request.urlopen(OFFICIAL_URL, timeout=120) as response:\n        # Reject downgrade or a redirect to an unrelated distribution host.\n        final_url = response.geturl()\n        if not final_url.startswith("https://snap.stanford.edu/ogb/"):\n            raise ValueError(f"Unexpected download redirect: {final_url}")\n        expected_bytes = int(response.headers.get("Content-Length", 0))\n        if expected_bytes:\n            print(f"Archive size: {expected_bytes / 2**30:.2f} GiB", flush=True)\n            if shutil.disk_usage(target_dir).free < expected_bytes + 4 * 2**30:\n                raise OSError("Not enough free disk for archive, train/valid files and cache; need archive size plus 4 GiB reserve")\n        with partial.open("xb") as stream:\n            downloaded = 0\n            next_progress = 128 * 1024 * 1024\n            while True:\n                block = response.read(8 * 1024 * 1024)\n                if not block:\n                    break\n                stream.write(block)\n                downloaded += len(block)\n                if downloaded >= next_progress:\n                    print(f"Downloaded {downloaded / 2**30:.2f} GiB", flush=True)\n                    next_progress += 128 * 1024 * 1024\n            if expected_bytes and downloaded != expected_bytes:\n                raise OSError("Download length mismatch; partial file retained, no extraction performed")\n    if archive.exists():\n        raise FileExistsError(archive)\n    # The target is a new private UUID directory, not a shared existing path.\n    os.rename(partial, archive)\n    print("Extracting only train, valid and small metadata (not test).", flush=True)\n    wanted = {\n        "biokg/split/random/train.pt", "biokg/split/random/valid.pt",\n        "biokg/raw/num-node-dict.csv.gz", "biokg/RELEASE_v1.txt",\n    }\n    required = {"biokg/split/random/train.pt", "biokg/split/random/valid.pt"}\n    seen = set()\n    with zipfile.ZipFile(archive) as bundle:\n        for member in bundle.infolist():\n            # Validate every archive path even though only allowlisted files\n            # are extracted. No ZIP-controlled path is passed to extractall.\n            pure = PurePosixPath(member.filename)\n            mode = member.external_attr >> 16\n            if (pure.is_absolute() or ".." in pure.parts or "\\\\" in member.filename\n                    or ":" in member.filename or stat.S_ISLNK(mode)):\n                raise ValueError(f"Unsafe ZIP member: {member.filename}")\n            if member.filename not in wanted:\n                continue\n            if member.filename in seen or member.is_dir():\n                raise ValueError(f"Duplicate/non-file selected ZIP member: {member.filename}")\n            seen.add(member.filename)\n            output = _no_links(target_dir.joinpath(*pure.parts))\n            output.parent.mkdir(parents=True, exist_ok=True)\n            with bundle.open(member) as source, output.open("xb") as stream:\n                shutil.copyfileobj(source, stream, length=8 * 1024 * 1024)\n    if not required.issubset(seen):\n        raise ValueError(f"Official ZIP missing required members: {sorted(required - seen)}")\n    _write_json_new(target_dir / "download_provenance.json", {\n        "url": OFFICIAL_URL, "final_url": final_url, "retrieved_at_utc": _utc(),\n        "archive_sha256": _sha256(archive), "archive_bytes": archive.stat().st_size,\n        "extracted_members": sorted(seen), "test_extracted": False,\n    })\n    return target_dir / "biokg"\n\n\ndef _node_metadata(root: Path) -> dict:\n    path = _no_links(root / "raw" / "num-node-dict.csv.gz")\n    if not path.is_file():\n        return {"status": "not_available", "sizes_source": "fixed SPEC mapping"}\n    with gzip.open(path, "rt", encoding="utf-8", newline="") as stream:\n        reader = csv.DictReader(stream)\n        rows = list(reader)\n    if len(rows) != 1 or set(rows[0]) != set(TYPE_NAMES):\n        raise ValueError("num-node-dict must contain one graph and exactly the five expected types")\n    actual = {name: int(rows[0][name]) for name in TYPE_NAMES}\n    expected = dict(zip(TYPE_NAMES, TYPE_SIZES.tolist()))\n    if actual != expected:\n        raise ValueError(f"Node-size metadata mismatch: {actual}; expected {expected}")\n    return {"status": "verified", "num_nodes_dict": actual, "sha256": _sha256(path)}\n\n\ndef _as_array(value) -> np.ndarray:\n    if hasattr(value, "detach"):\n        value = value.detach().cpu().numpy()\n    return np.asarray(value)\n\n\ndef _integer(name: str, values: np.ndarray, shape: tuple, low, high) -> None:\n    if values.shape != shape or values.dtype.kind not in "iu":\n        raise ValueError(f"{name}: expected integer shape {shape}, got {values.shape}/{values.dtype}")\n    if values.size and (np.any(values < low) or np.any(values >= high)):\n        raise ValueError(f"{name}: values outside the allowed typed range")\n\n\ndef _type_ids(name: str, value, size: int) -> np.ndarray:\n    array = _as_array(value)\n    if array.shape != (size,):\n        raise ValueError(f"{name}: incorrect type-label shape {array.shape}")\n    if array.dtype.kind in "iu":\n        _integer(name, array, (size,), 0, 5)\n        return array.astype(np.uint8)\n    # Public OGB stores type labels as strings. Only five vector comparisons,\n    # never one Python object/string per edge in the output cache.\n    out = np.full(size, 255, dtype=np.uint8)\n    for type_id, type_name in enumerate(TYPE_NAMES):\n        out[array == type_name] = type_id\n    if np.any(out == 255):\n        raise ValueError(f"{name}: unknown type label")\n    return out\n\n\ndef _npy_new(path: Path, dtype, shape: tuple) -> np.memmap:\n    # Exclusive reservation prevents accidentally overwriting any cache object.\n    with path.open("xb"):\n        pass\n    return np.lib.format.open_memmap(path, mode="w+", dtype=dtype, shape=shape)\n\n\ndef _prepare_split(root: Path, cache: Path, split: str, source_hash: str) -> dict:\n    import torch  # loading is explicit; no torch import or work at module import\n\n    print(f"Preparing {split}: trusted public .pt -> typed mmap cache", flush=True)\n\n    source = _no_links(root / "split" / "random" / f"{split}.pt")\n    # Pickle is required for public OGB NumPy dictionaries. Callers supplying\n    # INPUT_DIR explicitly assert these are trusted public OGB split files.\n    raw = torch.load(source, weights_only=False, map_location="cpu")\n    if not isinstance(raw, dict):\n        raise ValueError(f"{split}.pt is not an OGB split dictionary")\n    required = {"head", "relation", "tail", "head_type", "tail_type"}\n    if split == "valid":\n        required.update(("head_neg", "tail_neg"))\n    if not required.issubset(raw):\n        raise ValueError(f"{split}: missing keys {sorted(required - set(raw))}")\n    n_rows = EXPECTED_COUNTS[split]\n    head_type = _type_ids(f"{split}.head_type", raw["head_type"], n_rows)\n    tail_type = _type_ids(f"{split}.tail_type", raw["tail_type"], n_rows)\n    folder = cache / split\n    folder.mkdir(exist_ok=False)\n    metadata = {"n_rows": n_rows, "source_sha256": source_hash, "arrays": {}}\n    for name, types in (("head", head_type), ("tail", tail_type), ("relation", None)):\n        original = _as_array(raw[name])\n        upper = N_RELATIONS if types is None else TYPE_SIZES[types]\n        _integer(f"{split}.{name}", original, (n_rows,), 0, upper)\n        dtype = np.uint16 if types is None else np.uint32\n        result = _npy_new(folder / f"{name}.npy", dtype, (n_rows,))\n        for begin in range(0, n_rows, CHUNK_ROWS):\n            end = min(n_rows, begin + CHUNK_ROWS)\n            block = original[begin:end].astype(np.int64)\n            if types is not None:\n                block += TYPE_OFFSETS[types[begin:end]]\n            result[begin:end] = block\n            if not np.array_equal(result[begin:end].astype(np.int64), block):\n                raise ValueError(f"{split}.{name}: lossy dtype conversion")\n        result.flush()\n        metadata["arrays"][name] = {"source_dtype": str(original.dtype), "dtype": str(result.dtype), "shape": list(result.shape)}\n        del result\n    for name, values in (("head_type", head_type), ("tail_type", tail_type),\n                         ("row_id", np.arange(n_rows, dtype=np.uint32))):\n        result = _npy_new(folder / f"{name}.npy", values.dtype, values.shape)\n        result[:] = values\n        result.flush()\n        metadata["arrays"][name] = {"dtype": str(values.dtype), "shape": list(values.shape)}\n        if name != "row_id":\n            metadata["arrays"][name]["source_dtype"] = str(_as_array(raw[name]).dtype)\n        del result\n    if split == "valid":\n        for name, types in (("head_neg", head_type), ("tail_neg", tail_type)):\n            original = _as_array(raw[name])\n            if original.shape != (n_rows, 500) or original.dtype.kind not in "iu":\n                raise ValueError(f"{split}.{name}: expected integer [{n_rows},500]")\n            result = _npy_new(folder / f"{name}.npy", np.uint32, original.shape)\n            for begin in range(0, n_rows, CHUNK_ROWS):\n                end = min(n_rows, begin + CHUNK_ROWS)\n                block = original[begin:end]\n                bounds = TYPE_SIZES[types[begin:end], None]\n                _integer(f"{split}.{name}", block, (end - begin, 500), 0, bounds)\n                global_block = block.astype(np.int64) + TYPE_OFFSETS[types[begin:end], None]\n                result[begin:end] = global_block\n                if not np.array_equal(result[begin:end].astype(np.int64), global_block):\n                    raise ValueError(f"{split}.{name}: lossy uint32 conversion")\n            result.flush()\n            metadata["arrays"][name] = {\n                "source_dtype": str(original.dtype), "dtype": "uint32", "shape": list(original.shape),\n                "source_ids": "type-local, retained unchanged in original .pt",\n                "cache_ids": "global; recover local by subtracting row\'s type offset",\n                "order_preserved": True, "duplicates_preserved": True,\n            }\n            del result\n    # Hash cached values too: a completed manifest is the cache commit marker.\n    for name, entry in metadata["arrays"].items():\n        entry["sha256"] = _sha256(folder / f"{name}.npy")\n    del raw\n    return metadata\n\n\ndef _schema(train: dict, valid: dict) -> tuple[np.ndarray, np.ndarray]:\n    heads = np.full(N_RELATIONS, -1, dtype=np.int64)\n    tails = np.full(N_RELATIONS, -1, dtype=np.int64)\n    for split in (train, valid):\n        for relation in range(N_RELATIONS):\n            selected = split["relation"] == relation\n            if not np.any(selected):\n                continue\n            pairs = np.unique(np.stack((split["head_type"][selected], split["tail_type"][selected]), axis=1), axis=0)\n            if len(pairs) != 1:\n                raise ValueError(f"Relation {relation} has multiple type schemas")\n            head, tail = map(int, pairs[0])\n            if heads[relation] >= 0 and (heads[relation] != head or tails[relation] != tail):\n                raise ValueError(f"Relation {relation}: train/valid schema disagreement")\n            heads[relation], tails[relation] = head, tail\n    if np.any(heads < 0):\n        raise ValueError("Some expected relations have no schema in train/valid")\n    return heads, tails\n\n\ndef load_data(workspace: Path, input_dir: Path | None = None, download: bool = False) -> dict:\n    """Load public TRAIN+VALID only; downloading requires download=True.\n\n    INPUT_DIR is an explicit trust assertion for these two public pickle files.\n    Existing complete caches are verified and mmap-loaded. Incomplete artifacts\n    are preserved; the next call creates a fresh UUID cache without cleanup.\n    """\n    workspace = _no_links(Path(workspace))\n    workspace.mkdir(parents=True, exist_ok=True)\n    data_dir = _no_links(workspace / "data")\n    data_dir.mkdir(exist_ok=True)\n    pointer = _no_links(data_dir / "public_source.json")\n    if input_dir is not None:\n        root = _find_public_root(Path(input_dir))\n        provenance = {"kind": "explicit_trusted_public_ogb_input", "network_download_this_call": False}\n    elif pointer.is_file():\n        selection = json.loads(pointer.read_text(encoding="utf-8"))\n        relative = PurePosixPath(selection["relative_root"])\n        if relative.is_absolute() or ".." in relative.parts or ":" in str(relative) or "\\\\" in str(relative):\n            raise ValueError("Invalid persisted public source path")\n        root = _find_public_root(data_dir.joinpath(*relative.parts))\n        provenance = {"kind": "previous_explicit_public_download", "network_download_this_call": False}\n    elif download:\n        root = _download_public(data_dir)\n        _write_json_new(pointer, {"relative_root": root.relative_to(data_dir).as_posix()})\n        provenance = {"kind": "official_https_download", "network_download_this_call": True}\n    else:\n        raise FileNotFoundError("Provide trusted public OGB INPUT_DIR or explicitly set DOWNLOAD_DATA=True. No download was attempted.")\n    node_metadata = _node_metadata(root)\n    sources = {s: {"sha256": _sha256(root / "split" / "random" / f"{s}.pt"),\n                   "bytes": (root / "split" / "random" / f"{s}.pt").stat().st_size}\n               for s in ("train", "valid")}\n    fingerprint = hashlib.sha256(json.dumps(sources, sort_keys=True).encode()).hexdigest()\n    # Exact pointer lookup avoids enumerating arbitrary workspace directories.\n    cache_pointer = _no_links(data_dir / f"cache_v{CACHE_VERSION}_{fingerprint}.json")\n    if cache_pointer.is_file():\n        cache_name = json.loads(cache_pointer.read_text(encoding="utf-8"))["cache_directory"]\n        if Path(cache_name).name != cache_name:\n            raise ValueError("Invalid cache pointer")\n        cache = _no_links(data_dir / cache_name)\n        manifest = json.loads(_no_links(cache / "manifest.json").read_text(encoding="utf-8"))\n        if manifest["source_files"] != sources or manifest["cache_version"] != CACHE_VERSION:\n            raise ValueError("Cached source manifest mismatch")\n    else:\n        cache = _no_links(data_dir / f"cache_v{CACHE_VERSION}_{fingerprint[:16]}_{uuid.uuid4().hex}")\n        cache.mkdir(exist_ok=False)\n        split_metadata = {s: _prepare_split(root, cache, s, sources[s]["sha256"]) for s in ("train", "valid")}\n        mapping = _mapping()\n        manifest = {\n            "dataset": "ogbl-biokg", "dataset_release_expected": "1",\n            "cache_version": CACHE_VERSION, "created_at_utc": _utc(),\n            "official_url": OFFICIAL_URL, "source_provenance": provenance,\n            "source_files": sources, "source_references": _SOURCE_REFS,\n            "packages": {p: _version(p) for p in ("ogb", "torch", "numpy")},\n            "node_metadata": node_metadata, "splits": split_metadata,\n            "expected_split_counts": EXPECTED_COUNTS,\n            "mapping_sha256": hashlib.sha256(json.dumps(mapping, sort_keys=True).encode()).hexdigest(),\n            "n_negative": 500, "test_opened": False,\n            "test_overlap": "not_checked", "all_known_negative_filter": "not_verified_test_not_loaded",\n            "stage": "preliminary_validation_only",\n        }\n        download_provenance = root.parent / "download_provenance.json"\n        if input_dir is None and download_provenance.is_file():\n            manifest["download"] = json.loads(download_provenance.read_text(encoding="utf-8"))\n        _write_json_new(cache / "entity_mapping.json", mapping)\n        _write_json_new(cache / "manifest.json", manifest)\n        _write_json_new(cache_pointer, {"cache_directory": cache.name})\n    splits = {}\n    for split in ("train", "valid"):\n        splits[split] = {}\n        columns = _SPLIT_COLUMNS + (("head_neg", "tail_neg") if split == "valid" else ())\n        for column in columns:\n            path = _no_links(cache / split / f"{column}.npy")\n            entry = manifest["splits"][split]["arrays"][column]\n            if _sha256(path) != entry["sha256"]:\n                raise ValueError(f"Cached array hash mismatch: {split}.{column}")\n            array = np.load(path, mmap_mode="r", allow_pickle=False)\n            if str(array.dtype) != entry["dtype"] or list(array.shape) != entry["shape"]:\n                raise ValueError(f"Cached array metadata mismatch: {split}.{column}")\n            splits[split][column] = array\n    schema_head, schema_tail = _schema(splits["train"], splits["valid"])\n    return {**splits, "n_entities": N_ENTITIES, "n_relations": N_RELATIONS,\n            "type_offsets": TYPE_OFFSETS.copy(), "type_sizes": TYPE_SIZES.copy(),\n            "type_names": TYPE_NAMES, "schema_head": schema_head, "schema_tail": schema_tail,\n            "mapping": _mapping(), "manifest": manifest, "cache_dir": cache}\n\n\ndef triple_keys(heads, relations, tails, n_entities: int = N_ENTITIES, n_relations: int = N_RELATIONS) -> np.ndarray:\n    """Collision-free uint64 key, after callers have checked index ranges."""\n    if n_entities * n_relations * n_entities - 1 > np.iinfo(np.uint64).max:\n        raise OverflowError("Triple keys do not fit uint64")\n    return ((np.asarray(heads, dtype=np.uint64) * np.uint64(n_relations)\n             + np.asarray(relations, dtype=np.uint64)) * np.uint64(n_entities)\n            + np.asarray(tails, dtype=np.uint64))\n\n\ndef contains_keys(sorted_keys: np.ndarray, query_keys: np.ndarray) -> np.ndarray:\n    """Vectorized membership without Python tuples or sets of graph edges."""\n    if not len(sorted_keys):\n        return np.zeros(query_keys.shape, dtype=bool)\n    positions = np.searchsorted(sorted_keys, query_keys)\n    bounded = np.minimum(positions, len(sorted_keys) - 1)\n    return (positions < len(sorted_keys)) & (sorted_keys[bounded] == query_keys)\n\n\ndef audit_data(data: dict, output_dir: Path) -> dict:\n    """Run required first-stage gates; emit JSON before raising on failure.\n\n    Known positives here mean TRAIN union VALID only. TEST and all-known\n    filtering are explicitly unverified; success never means final-test ready.\n    """\n    output_dir = _no_links(Path(output_dir))\n    print("Auditing train/valid IDs, overlap and official negatives on CPU.", flush=True)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    output = output_dir / "data_integrity.json"\n    if output.exists():\n        raise FileExistsError("Choose a fresh run output directory; audit reports are never overwritten")\n    report = {"stage": "preliminary_validation_only", "status": "running",\n              "test_opened": False, "test_overlap": "not_checked",\n              "all_known_negative_filter": "not_verified_test_not_loaded", "splits": {}}\n    try:\n        if data["n_entities"] != N_ENTITIES or data["n_relations"] != N_RELATIONS:\n            raise ValueError("Dataset entity/relation cardinality differs from SPEC")\n        if not np.array_equal(data["type_sizes"], TYPE_SIZES) or not np.array_equal(data["type_offsets"], TYPE_OFFSETS):\n            raise ValueError("Typed global ID mapping differs from SPEC")\n        report["mapping_roundtrip"] = "passed; disjoint contiguous type intervals"\n        if not np.array_equal(TYPE_OFFSETS[1:], TYPE_OFFSETS[:-1] + TYPE_SIZES[:-1]) or int(TYPE_OFFSETS[-1] + TYPE_SIZES[-1]) != N_ENTITIES:\n            raise ValueError("Global mapping intervals overlap or leave gaps")\n        indices = {}\n        for split_name in ("train", "valid"):\n            split = data[split_name]\n            n = EXPECTED_COUNTS[split_name]\n            for name in _SPLIT_COLUMNS:\n                upper = 5 if name.endswith("type") else N_RELATIONS if name == "relation" else n if name == "row_id" else N_ENTITIES\n                _integer(f"{split_name}.{name}", split[name], (n,), 0, upper)\n            if not np.array_equal(split["row_id"], np.arange(n, dtype=np.uint32)):\n                raise ValueError(f"{split_name}: original row order has changed")\n            if split["head_type"].dtype != np.uint8 or split["tail_type"].dtype != np.uint8:\n                raise ValueError(f"{split_name}: expected uint8 type IDs")\n            for role in ("head", "tail"):\n                types = split[f"{role}_type"]\n                offsets = TYPE_OFFSETS[types]\n                _integer(f"{split_name}.{role}", split[role], (n,), offsets, offsets + TYPE_SIZES[types])\n                if not np.array_equal(split[f"{role}_type"], data[f"schema_{role}"][split["relation"]]):\n                    raise ValueError(f"{split_name}: relation schema mismatch")\n            keys = np.sort(triple_keys(split["head"], split["relation"], split["tail"]))\n            duplicate_count = int(np.count_nonzero(keys[1:] == keys[:-1]))\n            indices[split_name] = np.unique(keys)\n            report["splits"][split_name] = {\n                "n_rows": n, "expected_count_gate": "passed", "duplicate_rows": duplicate_count,\n                "dtype_and_range_gate": "passed", "relation_counts": np.bincount(split["relation"], minlength=N_RELATIONS).tolist(),\n                "dtypes": {k: str(split[k].dtype) for k in _SPLIT_COLUMNS},\n            }\n        overlap = int(np.intersect1d(indices["train"], indices["valid"], assume_unique=True).size)\n        report["direct_train_valid_overlap_unique_triples"] = overlap\n        if overlap:\n            raise ValueError("Exact train/valid positive overlap: evaluation is on hold")\n        # This index stays in this audit function and is never returned to a trainer.\n        audit_known = np.union1d(indices["train"], indices["valid"])\n        valid = data["valid"]\n        for role in ("head", "tail"):\n            negative = valid[f"{role}_neg"]\n            print(f"Auditing {role} negatives (all 500 per validation row).", flush=True)\n            n = EXPECTED_COUNTS["valid"]\n            if negative.shape != (n, 500) or negative.dtype.kind not in "iu":\n                raise ValueError(f"valid.{role}_neg: expected integer [{n},500]")\n            duplicates = 0\n            hit_count = 0\n            for begin in range(0, n, CHUNK_ROWS):\n                end = min(n, begin + CHUNK_ROWS)\n                block = negative[begin:end]\n                types = valid[f"{role}_type"][begin:end]\n                lo = TYPE_OFFSETS[types, None]\n                _integer(f"valid.{role}_neg", block, block.shape, lo, lo + TYPE_SIZES[types, None])\n                local = block.astype(np.int64) - lo\n                if not np.array_equal(local + lo, block):\n                    raise ValueError("Negative global/local round-trip failed")\n                ordered = np.sort(block, axis=1)\n                duplicates += int(np.count_nonzero(ordered[:, 1:] == ordered[:, :-1]))\n                relations = valid["relation"][begin:end, None]\n                fixed = valid["tail" if role == "head" else "head"][begin:end, None]\n                candidate_keys = triple_keys(block, relations, fixed) if role == "head" else triple_keys(fixed, relations, block)\n                hit_count += int(np.count_nonzero(contains_keys(audit_known, candidate_keys)))\n            report["splits"]["valid"][f"{role}_negative_audit"] = {\n                "shape": list(negative.shape), "dtype": str(negative.dtype),\n                "duplicate_entries_beyond_first_per_row": duplicates,\n                "train_or_valid_positive_collisions": hit_count,\n                "order_preserved": True, "duplicates_preserved": True,\n            }\n            if hit_count:\n                raise ValueError(f"Official {role} negatives contain TRAIN/VALID positives; source investigation required")\n        report["schema"] = {"head_type": data["schema_head"].tolist(), "tail_type": data["schema_tail"].tolist()}\n        report["status"] = "passed_first_stage_only"\n        report["required_first_stage_gates_passed"] = True\n    except Exception as error:\n        report["status"] = "failed"\n        report["required_first_stage_gates_passed"] = False\n        report["error"] = f"{type(error).__name__}: {error}"\n        _write_json_new(output, report)\n        raise\n    _write_json_new(output, report)\n    return report\n', 'baselines.py': '"""Train-only statistics and cheap candidate scorers.\n\nThe scorer input has row_id[B], scalar direction (head=0, tail=1), relation[B],\nfixed[B], target_type[B]. It contains no target or positive flag. Candidate\nmatrices are GLOBAL IDs [B,K]; larger scores mean a better candidate.\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\n\nfrom .data import contains_keys, triple_keys\n\n\ndef _train_arrays(data: dict) -> tuple:\n    train = data["train"]\n    return tuple(np.asarray(train[k], dtype=np.int64) for k in ("head", "relation", "tail"))\n\n\ndef prepare_statistics(data: dict) -> dict:\n    """Return count tables, compact TRAIN index, and train-universe thresholds.\n\n    Validation/test labels are never read. Schema is inferred independently\n    from train type IDs, rather than using potentially held-out schema labels.\n    Count arrays include zeros for every entity and retain duplicate train rows.\n    """\n    n, r = int(data["n_entities"]), int(data["n_relations"])\n    train = data["train"]\n    heads, relations, tails = _train_arrays(data)\n    sizes = np.asarray(data["type_sizes"], dtype=np.int64)\n    offsets = np.asarray(data["type_offsets"], dtype=np.int64)\n    counts = np.zeros((r, 2, n), dtype=np.int64)\n    np.add.at(counts[:, 0, :], (relations, heads), 1)\n    np.add.at(counts[:, 1, :], (relations, tails), 1)\n    schema = np.full((r, 2), -1, dtype=np.int64)\n    thresholds = np.zeros((r, 2, 2), dtype=np.int64)\n    cells = []\n    for relation in range(r):\n        selected = relations == relation\n        if not np.any(selected):\n            raise ValueError(f"Relation {relation}: no train schema; held-out schema cannot fill it")\n        for direction, role in ((0, "head"), (1, "tail")):\n            types = np.unique(train[f"{role}_type"][selected])\n            if len(types) != 1:\n                raise ValueError(f"Relation {relation}, {role}: multiple train schemas")\n            type_id = int(types[0])\n            schema[relation, direction] = type_id\n            start, stop = int(offsets[type_id]), int(offsets[type_id] + sizes[type_id])\n            universe = counts[relation, direction, start:stop]\n            positive = np.sort(universe[universe > 0])\n            q1 = int(positive[(len(positive) + 2) // 3 - 1]) if len(positive) else 0\n            q2 = int(positive[(2 * len(positive) + 2) // 3 - 1]) if len(positive) else 0\n            thresholds[relation, direction] = (q1, q2)\n            total = int(universe.sum(dtype=np.int64))\n            ordered = np.sort(universe)\n            ranks = np.arange(1, len(ordered) + 1, dtype=np.float64)\n            gini = float(2 * np.dot(ranks, ordered.astype(np.float64)) / (len(ordered) * total)\n                         - (len(ordered) + 1) / len(ordered)) if total else None\n            cells.append({"relation": relation, "direction": direction, "target_type": type_id,\n                          "universe_size": len(universe), "nonzero_entities": int(len(positive)),\n                          "zero_fraction": float(np.mean(universe == 0)), "gini": gini,\n                          "q1": q1, "q2": q2, "train_role_rows": total})\n    train_keys = np.unique(triple_keys(heads, relations, tails, n, r))\n    same_type = schema[:, 0] == schema[:, 1]\n    reverse = contains_keys(train_keys, triple_keys(tails, relations, heads, n, r)) & same_type[relations]\n    reverse_rows = []\n    for relation in range(r):\n        selected = relations == relation\n        self_loop = selected & (heads == tails)\n        nonself = selected & (heads != tails)\n        applicable = bool(same_type[relation])\n        denominator = int(np.count_nonzero(nonself))\n        reverse_rows.append({"relation": relation, "applicable": applicable,\n                             "n_train_rows": int(np.count_nonzero(selected)),\n                             "self_loop_rows": int(np.count_nonzero(self_loop)),\n                             "nonself_train_rows": denominator,\n                             "reverse_rows": int(np.count_nonzero(reverse & selected)) if applicable else None,\n                             "reverse_nonself_fraction": float(np.count_nonzero(reverse & nonself) / denominator) if applicable and denominator else None})\n    return {"counts": counts, "train_keys": train_keys, "same_type": same_type,\n            "schema_head": schema[:, 0], "schema_tail": schema[:, 1],\n            "thresholds": thresholds, "bins": ("zero", "low", "mid", "high"),\n            "frequency_cells": cells, "reverse_diagnostics": reverse_rows,\n            "type_offsets": offsets.copy(), "type_sizes": sizes.copy(),\n            "n_entities": n, "n_relations": r,\n            "statistics_source": "train_only; duplicate original rows retained for counts"}\n\n\ndef _query(query: dict, candidates: np.ndarray) -> tuple:\n    forbidden = {"target", "target_global", "positive", "positive_flag", "frequency_bin", "inverse_available_positive"}\n    if forbidden.intersection(query):\n        raise ValueError("Scorers must not receive labels or diagnostic features")\n    candidates = np.asarray(candidates)\n    if candidates.ndim != 2 or candidates.dtype.kind not in "iu":\n        raise ValueError("Candidates must be integer GLOBAL IDs [B,K]")\n    direction = query["direction"]\n    if np.ndim(direction) != 0 or int(direction) not in (0, 1):\n        raise ValueError("Each scorer batch must have one scalar direction (0=head, 1=tail)")\n    size = candidates.shape[0]\n    values = []\n    for name in ("row_id", "relation", "fixed", "target_type"):\n        array = np.asarray(query[name])\n        if array.shape != (size,) or array.dtype.kind not in "iu":\n            raise ValueError(f"query.{name} must be integer [B]")\n        values.append(array.astype(np.int64, copy=False))\n    return (int(direction), *values, candidates.astype(np.int64, copy=False))\n\n\ndef _typed_candidates(stats: dict, direction, relations, fixed, types, candidates) -> None:\n    if np.any(relations < 0) or np.any(relations >= stats["n_relations"]):\n        raise ValueError("Query relation outside train schema")\n    expected_type = stats["schema_head" if direction == 0 else "schema_tail"][relations]\n    if not np.array_equal(types, expected_type):\n        raise ValueError("Query target type disagrees with train-only schema")\n    low = stats["type_offsets"][types, None]\n    high = low + stats["type_sizes"][types, None]\n    if np.any(candidates < low) or np.any(candidates >= high):\n        raise ValueError("Candidate GLOBAL ID has the wrong endpoint type")\n    fixed_types = stats["schema_tail" if direction == 0 else "schema_head"][relations]\n    fixed_low = stats["type_offsets"][fixed_types]\n    if np.any(fixed < fixed_low) or np.any(fixed >= fixed_low + stats["type_sizes"][fixed_types]):\n        raise ValueError("Fixed GLOBAL ID has the wrong endpoint type")\n\n\nclass CountPrior:\n    """Count of candidate in the requested relation/role in TRAIN, int64."""\n\n    def __init__(self, stats: dict):\n        self.stats = stats\n\n    def score_candidates(self, query: dict, candidates: np.ndarray) -> np.ndarray:\n        direction, _, relations, fixed, types, candidates = _query(query, candidates)\n        _typed_candidates(self.stats, direction, relations, fixed, types, candidates)\n        return self.stats["counts"][relations[:, None], direction, candidates]\n\n\nclass ReverseLookup:\n    """TRAIN reverse indicator + bounded count score; float64, no labels."""\n\n    def __init__(self, stats: dict, reverse_only: bool = False):\n        self.stats = stats\n        self.reverse_only = bool(reverse_only)\n\n    def score_candidates(self, query: dict, candidates: np.ndarray) -> np.ndarray:\n        direction, _, relations, fixed, types, candidates = _query(query, candidates)\n        _typed_candidates(self.stats, direction, relations, fixed, types, candidates)\n        # For (?,r,t), candidate reverse is (t,r,c); for (h,r,?),\n        # candidate reverse is (c,r,h). Cross-type swaps are inapplicable.\n        keys = (triple_keys(fixed[:, None], relations[:, None], candidates,\n                            self.stats["n_entities"], self.stats["n_relations"]) if direction == 0\n                else triple_keys(candidates, relations[:, None], fixed[:, None],\n                                 self.stats["n_entities"], self.stats["n_relations"]))\n        indicator = contains_keys(self.stats["train_keys"], keys)\n        indicator &= self.stats["same_type"][relations, None]\n        score = indicator.astype(np.float64)\n        if not self.reverse_only:\n            counts = self.stats["counts"][relations[:, None], direction, candidates].astype(np.float64)\n            score += 0.25 * counts / (counts + 1.0)\n        return score\n\n\ndef _mix64(value: np.ndarray) -> np.ndarray:\n    """SplitMix64 finalizer; arithmetic modulo 2**64 is intentional.\n\n    Algorithm constants: Steele, Lea, Flood, SPLIT/OOPSLA 2014;\n    https://prng.di.unimi.it/splitmix64.c (public-domain reference).\n    """\n    with np.errstate(over="ignore"):\n        value = np.asarray(value, dtype=np.uint64).copy()\n        value ^= value >> np.uint64(30)\n        value *= np.uint64(0xBF58476D1CE4E5B9)\n        value ^= value >> np.uint64(27)\n        value *= np.uint64(0x94D049BB133111EB)\n        value ^= value >> np.uint64(31)\n    return value\n\n\nclass RandomScorer:\n    """Stable candidate-specific float64 [0,1), independent of column order.\n\n    The optional split defaults to valid for this validation-only backend.\n    There is no Python hash, RNG state, target argument, or special column 0.\n    Identical candidates in one query receive identical scores.\n    """\n\n    def __init__(self, seed: int = 101):\n        if not isinstance(seed, (int, np.integer)) or not 0 <= int(seed) < 2**64:\n            raise ValueError("Random seed must fit uint64")\n        self.seed = int(seed)\n\n    def score_candidates(self, query: dict, candidates: np.ndarray) -> np.ndarray:\n        direction, rows, _, _, _, candidates = _query(query, candidates)\n        split_codes = {"train": 1, "valid": 2, "test": 3}\n        split = query.get("split", "valid")\n        if split not in split_codes:\n            raise ValueError("Random scorer needs a stable known split name")\n        if np.any(rows < 0) or np.any(candidates < 0):\n            raise ValueError("Rows and GLOBAL candidate IDs must be nonnegative")\n        # Stable serialization is a sequence of uint64 fields mixed with\n        # separate domain tags, not ambiguous decimal/string concatenation.\n        seed_key = _mix64(np.array([self.seed], dtype=np.uint64))[0]\n        row_key = _mix64(rows.astype(np.uint64) ^ np.uint64(0x243F6A8885A308D3))\n        domain = np.uint64(split_codes[split] * 2 + direction)\n        query_key = _mix64(row_key ^ seed_key ^ _mix64(np.array([domain]))[0])\n        mixed = _mix64(query_key[:, None] ^ _mix64(candidates.astype(np.uint64) ^ np.uint64(0x13198A2E03707344)))\n        return (mixed >> np.uint64(11)).astype(np.float64) * (1.0 / 2**53)\n\n\ndef classify_frequency(stats: dict, relations, direction: int, targets) -> np.ndarray:\n    """Analysis-only zero/low/mid/high labels using frozen TRAIN thresholds."""\n    if direction not in (0, 1):\n        raise ValueError("direction must be head=0 or tail=1")\n    relations = np.asarray(relations, dtype=np.int64)\n    targets = np.asarray(targets, dtype=np.int64)\n    if relations.shape != targets.shape or relations.ndim != 1:\n        raise ValueError("Relations and target IDs must be matching [B] arrays")\n    counts = stats["counts"][relations, direction, targets]\n    q1 = stats["thresholds"][relations, direction, 0]\n    q2 = stats["thresholds"][relations, direction, 1]\n    labels = np.full(counts.shape, "high", dtype="<U4")\n    labels[counts <= q2] = "mid"\n    labels[counts <= q1] = "low"\n    labels[counts == 0] = "zero"\n    return labels\n\n\ndef reverse_available(stats: dict, relations, heads, tails) -> tuple[np.ndarray, np.ndarray]:\n    """Analysis-only (applicable, available) booleans for positive triples.\n\n    Available=False for cross-type relations is masked by applicable=False;\n    exported tables must represent that diagnostic stratum as N/A.\n    """\n    relations = np.asarray(relations, dtype=np.int64)\n    heads, tails = np.asarray(heads, dtype=np.int64), np.asarray(tails, dtype=np.int64)\n    if relations.ndim != 1 or heads.shape != relations.shape or tails.shape != relations.shape:\n        raise ValueError("Relations, heads, tails must be matching [B] arrays")\n    applicable = stats["same_type"][relations]\n    available = contains_keys(stats["train_keys"], triple_keys(tails, relations, heads,\n                              stats["n_entities"], stats["n_relations"])) & applicable\n    return applicable, available\n', 'evaluate.py': '"""Validation-only, chunked evaluation for the first Kaggle pilot.\n\nAll functions are inert until explicitly called. This is not final-test code.\n"""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .baselines import classify_frequency, reverse_available\n\n\ndef select_validation_rows(data: dict, max_per_relation: int | None = 20,\n                           seed: int = 71) -> np.ndarray:\n    """The same original triple rows (both directions) for every scorer."""\n    relations = np.asarray(data[\'valid\'][\'relation\'])\n    if max_per_relation is None:\n        return np.arange(len(relations), dtype=np.int64)\n    if max_per_relation < 1:\n        raise ValueError(\'max_per_relation must be positive or None\')\n    rng = np.random.default_rng(seed)\n    selected = []\n    for relation in np.unique(relations):\n        rows = np.flatnonzero(relations == relation)\n        selected.extend(rng.choice(rows, size=min(max_per_relation, len(rows)),\n                                   replace=False).tolist())\n    return np.array(sorted(selected), dtype=np.int64)\n\n\ndef midrank_metrics(positive: np.ndarray, negatives: np.ndarray) -> dict:\n    """No dtype downcast, epsilon ties or random tie-breaking."""\n    positive = np.asarray(positive)\n    negatives = np.asarray(negatives)\n    if positive.ndim != 1 or negatives.ndim != 2 or len(positive) != len(negatives):\n        raise ValueError(\'Expected positive[B] and negatives[B,K]\')\n    if not np.isfinite(positive).all() or not np.isfinite(negatives).all():\n        raise ValueError(\'Non-finite scores: evaluation stopped\')\n    greater = (negatives > positive[:, None]).sum(axis=1, dtype=np.int64)\n    equal = (negatives == positive[:, None]).sum(axis=1, dtype=np.int64)\n    return _metrics_from_counts(greater, equal)\n\n\ndef _metrics_from_counts(greater, equal):\n    rank = 1.0 + greater + 0.5 * equal\n    return {\'n_greater\': greater, \'n_equal\': equal, \'rank\': rank,\n            \'rr\': 1.0 / rank, \'hits1\': rank <= 1,\n            \'hits3\': rank <= 3, \'hits10\': rank <= 10}\n\n\ndef metric_gates() -> dict:\n    """Future notebook gate against the INSTALLED ogb Evaluator, not a clone.\n\n    Calling this executes tiny synthetic checks only. It has not been called\n    during preparation of this notebook.\n    """\n    from ogb.linkproppred import Evaluator\n    import ogb\n    import ogb.linkproppred.evaluate as official_module\n\n    rng = np.random.default_rng(71)\n    positive = np.r_[3.0, -3.0, 0.0, rng.normal(size=29)]\n    negative = np.vstack([np.full(500, -2.0), np.full(500, 2.0),\n                          np.zeros(500), rng.normal(size=(29, 500))])\n    negative[3, :8] = positive[3]\n    ours = midrank_metrics(positive, negative)\n    reference = Evaluator(name=\'ogbl-biokg\').eval(\n        {\'y_pred_pos\': positive, \'y_pred_neg\': negative})\n    for local_key, reference_key in [(\'rr\', \'mrr_list\'), (\'hits1\', \'hits@1_list\'),\n                                      (\'hits3\', \'hits@3_list\'), (\'hits10\', \'hits@10_list\')]:\n        np.testing.assert_allclose(ours[local_key], reference[reference_key],\n                                   atol=1e-7, rtol=1e-7)\n    np.testing.assert_equal(ours[\'rank\'][:3], [1.0, 501.0, 251.0])\n    reordered = midrank_metrics(positive, negative[:, ::-1])\n    np.testing.assert_array_equal(ours[\'rank\'], reordered[\'rank\'])\n    source = Path(official_module.__file__).read_bytes()\n    return {\'status\': \'passed\', \'ogb_version\': ogb.__version__,\n            \'official_evaluator_sha256\': hashlib.sha256(source).hexdigest(),\n            \'checks\': [\'first_last_all_ties\', \'partial_ties\',\n                       \'numpy_official_parity\', \'candidate_permutation\']}\n\n\ndef baseline_gates(data: dict, stats: dict) -> dict:\n    """Small non-training checks on scorer invariants, run only on request."""\n    from .baselines import CountPrior, ReverseLookup, RandomScorer\n    rows = select_validation_rows(data, 1)[:8]\n    if not len(rows):\n        raise ValueError(\'Empty validation set\')\n    split = data[\'valid\']\n    query = {\'split\': \'valid\', \'row_id\': rows, \'direction\': 1,\n             \'relation\': np.asarray(split[\'relation\'][rows]),\n             \'fixed\': np.asarray(split[\'head\'][rows]),\n             \'target_type\': np.asarray(split[\'tail_type\'][rows])}\n    candidates = np.asarray(split[\'tail_neg\'][rows, :5], dtype=np.int64)\n    candidates[:, 1] = candidates[:, 0]\n    for scorer in [CountPrior(stats), ReverseLookup(stats), RandomScorer(101)]:\n        first = scorer.score_candidates(query, candidates)\n        reverse = scorer.score_candidates(query, candidates[:, ::-1])\n        np.testing.assert_array_equal(first, reverse[:, ::-1])\n        np.testing.assert_array_equal(first[:, 0], first[:, 1])\n    count = CountPrior(stats)\n    fixed_types = stats[\'schema_head\'][query[\'relation\']]\n    lower = stats[\'type_offsets\'][fixed_types]\n    sizes = stats[\'type_sizes\'][fixed_types]\n    changed_fixed = lower + (query[\'fixed\'] - lower + 1) % sizes\n    changed = dict(query, fixed=changed_fixed)\n    np.testing.assert_array_equal(count.score_candidates(query, candidates),\n                                  count.score_candidates(changed, candidates))\n    return {\'status\': \'passed\', \'rows_checked\': len(rows),\n            \'checks\': [\'candidate_permutation\', \'duplicate_id_consistency\',\n                       \'count_endpoint_blindness\']}\n\n\nclass _Aggregate:\n    def __init__(self):\n        self.count = 0\n        self.sums = np.zeros(4, dtype=np.float64)\n        self.ties = 0\n        self.clusters = set()\n        self.targets = set()\n\n    def add(self, row):\n        self.count += 1\n        self.sums += [row[\'rr\'], row[\'hits1\'], row[\'hits3\'], row[\'hits10\']]\n        self.ties += int(row[\'n_equal\'] > 0)\n        self.clusters.add(row[\'cluster_id\'])\n        self.targets.add(row[\'target_global\'])\n\n    def record(self, key):\n        means = self.sums / self.count\n        return dict(stratum=key, n_queries=self.count, n_clusters=len(self.clusters),\n                    n_unique_targets=len(self.targets), mrr=float(means[0]),\n                    hits1=float(means[1]), hits3=float(means[2]),\n                    hits10=float(means[3]), tie_query_fraction=self.ties/self.count,\n                    support=\'sufficient\' if len(self.clusters) >= 50 else \'small_cell\')\n\n\ndef evaluate_validation(scorer, data: dict, stats: dict, rows: np.ndarray,\n                        output_dir: Path | None = None, model_name: str = \'scorer\',\n                        training_seed: int | None = None, batch_size: int = 64,\n                        candidate_chunk: int = 128, save_queries: bool = True,\n                        verbose: bool = True) -> dict:\n    """Two queries per selected valid row, all 500 official negatives each.\n\n    score_candidates never receives the positive target or diagnostic labels.\n    Only chunks enter the GPU. There is deliberately no test argument.\n    """\n    if batch_size < 1 or candidate_chunk < 1:\n        raise ValueError(\'Batch and chunk must be positive\')\n    split = data[\'valid\']\n    rows = np.asarray(rows, dtype=np.int64)\n    if rows.ndim != 1 or not len(rows) or len(np.unique(rows)) != len(rows):\n        raise ValueError(\'Need nonempty unique validation row IDs\')\n    if rows.min() < 0 or rows.max() >= len(split[\'head\']):\n        raise ValueError(\'Validation row ID out of range\')\n    if split[\'head_neg\'].shape[1] != 500 or split[\'tail_neg\'].shape[1] != 500:\n        raise ValueError(\'Official evaluation requires 500 negatives per side\')\n    aggregates = defaultdict(_Aggregate)\n    candidate_hash = hashlib.sha256()\n    start = time.perf_counter()\n    query_handle = None\n    writer = None\n    columns = [\'split\', \'row_id\', \'direction\', \'relation\', \'model\', \'training_seed\',\n               \'head_global\', \'tail_global\', \'fixed_global\', \'target_global\',\n               \'target_type\', \'frequency_count\', \'frequency_bin\', \'inverse_applicable\',\n               \'inverse_available\', \'self_loop\', \'cluster_id\', \'n_negative\',\n               \'n_greater\', \'n_equal\', \'rank\', \'rr\', \'hits1\', \'hits3\', \'hits10\']\n    if output_dir is not None:\n        output_dir = Path(output_dir)\n        output_dir.mkdir(parents=True, exist_ok=False)\n        if save_queries:\n            query_handle = (output_dir / \'per_query.csv\').open(\'x\', encoding=\'utf-8\', newline=\'\')\n            writer = csv.DictWriter(query_handle, fieldnames=columns)\n            writer.writeheader()\n    try:\n        for direction in (0, 1):\n            target_key, fixed_key = (\'head\', \'tail\') if direction == 0 else (\'tail\', \'head\')\n            negative_key = \'head_neg\' if direction == 0 else \'tail_neg\'\n            for begin in range(0, len(rows), batch_size):\n                selected = rows[begin:begin+batch_size]\n                relation = np.asarray(split[\'relation\'][selected], dtype=np.int64)\n                head = np.asarray(split[\'head\'][selected], dtype=np.int64)\n                tail = np.asarray(split[\'tail\'][selected], dtype=np.int64)\n                targets = np.asarray(split[target_key][selected], dtype=np.int64)\n                fixed = np.asarray(split[fixed_key][selected], dtype=np.int64)\n                types = np.asarray(split[target_key+\'_type\'][selected], dtype=np.int64)\n                query = {\'split\': \'valid\', \'row_id\': selected, \'direction\': direction,\n                         \'relation\': relation, \'fixed\': fixed, \'target_type\': types}\n                positive = np.asarray(scorer.score_candidates(query, targets[:, None]))[:, 0]\n                if not np.isfinite(positive).all():\n                    raise ValueError(\'Non-finite positive score\')\n                greater = np.zeros(len(selected), dtype=np.int64)\n                equal = np.zeros(len(selected), dtype=np.int64)\n                # Canonical row-major serialization, independent of B/K chunks.\n                hash_negatives = np.asarray(split[negative_key][selected], dtype=\'<i8\')\n                for j, row_id in enumerate(selected):\n                    identity = np.array([row_id, direction, relation[j], fixed[j],\n                                         targets[j]], dtype=\'<i8\')\n                    candidate_hash.update(identity.tobytes())\n                    candidate_hash.update(hash_negatives[j].tobytes())\n                for offset in range(0, 500, candidate_chunk):\n                    candidates = np.asarray(split[negative_key][selected, offset:offset+candidate_chunk],\n                                            dtype=np.int64)\n                    scores = np.asarray(scorer.score_candidates(query, candidates))\n                    if scores.shape != candidates.shape or not np.isfinite(scores).all():\n                        raise ValueError(\'Invalid negative score array\')\n                    greater += (scores > positive[:, None]).sum(axis=1, dtype=np.int64)\n                    equal += (scores == positive[:, None]).sum(axis=1, dtype=np.int64)\n                metrics = _metrics_from_counts(greater, equal)\n                frequency = np.asarray(stats[\'counts\'][relation, direction, targets])\n                frequency_bins = classify_frequency(stats, relation, direction, targets)\n                applicable, inverse = reverse_available(stats, relation, head, tail)\n                for j, row_id in enumerate(selected):\n                    h, t, r = int(head[j]), int(tail[j]), int(relation[j])\n                    cluster = f\'{r}:{min(h,t)}:{max(h,t)}\' if applicable[j] else f\'{r}:{h}:{t}\'\n                    row = dict(split=\'valid\', row_id=int(row_id), direction=\'head\' if direction == 0 else \'tail\',\n                               relation=r, model=model_name, training_seed=training_seed,\n                               head_global=h, tail_global=t, fixed_global=int(fixed[j]),\n                               target_global=int(targets[j]), target_type=int(types[j]),\n                               frequency_count=int(frequency[j]), frequency_bin=str(frequency_bins[j]),\n                               inverse_applicable=bool(applicable[j]),\n                               inverse_available=bool(inverse[j]) if applicable[j] else None,\n                               self_loop=h==t, cluster_id=cluster, n_negative=500,\n                               n_greater=int(greater[j]), n_equal=int(equal[j]),\n                               rank=float(metrics[\'rank\'][j]), rr=float(metrics[\'rr\'][j]),\n                               hits1=int(metrics[\'hits1\'][j]), hits3=int(metrics[\'hits3\'][j]),\n                               hits10=int(metrics[\'hits10\'][j]))\n                    if writer:\n                        writer.writerow(row)\n                    strata = [\'all\', f"direction:{row[\'direction\']}",\n                              f"relation:{r}:{row[\'direction\']}",\n                              f"frequency:{row[\'frequency_bin\']}",\n                              f"relation_frequency:{r}:{row[\'direction\']}:{row[\'frequency_bin\']}"]\n                    if applicable[j]:\n                        strata.append(f\'reverse:{bool(inverse[j])}\')\n                        if h != t:\n                            strata.append(f\'reverse_nonself:{bool(inverse[j])}\')\n                    else:\n                        strata.append(\'reverse:not_applicable\')\n                    for stratum in strata:\n                        aggregates[stratum].add(row)\n                if verbose and (begin == 0 or begin+batch_size >= len(rows)):\n                    print(f\'{model_name}: {query["direction"]} / {min(begin+batch_size,len(rows))}/{len(rows)} triples\')\n    finally:\n        if query_handle:\n            query_handle.close()\n    records = [value.record(key) for key, value in sorted(aggregates.items())]\n    overall = aggregates[\'all\'].record(\'all\')\n    relation_means = [x[\'mrr\'] for x in records if x[\'stratum\'].startswith(\'relation:\')]\n    result = dict(stage=\'preliminary_validation\', split=\'valid\', model=model_name,\n                  training_seed=training_seed, n_triples=len(rows), n_queries=2*len(rows),\n                  mrr=overall[\'mrr\'], hits1=overall[\'hits1\'], hits3=overall[\'hits3\'],\n                  hits10=overall[\'hits10\'], macro_relation_role_mrr=float(np.mean(relation_means)),\n                  n_relation_role_cells=len(relation_means),\n                  official_negatives_per_direction=500,\n                  validation_rows_sha256=hashlib.sha256(rows.astype(\'<i8\').tobytes()).hexdigest(),\n                  candidate_sha256=candidate_hash.hexdigest(),\n                  evaluation_seconds=time.perf_counter()-start,\n                  note=\'Stratified validation subset; no full test/matched/CI claims.\')\n    if output_dir is not None:\n        (output_dir/\'summary.json\').write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding=\'utf-8\')\n        with (output_dir/\'strata.csv\').open(\'x\', encoding=\'utf-8\', newline=\'\') as handle:\n            table = csv.DictWriter(handle, fieldnames=list(records[0]))\n            table.writeheader()\n            table.writerows(records)\n    return result\n', 'training.py': '"""Cloud-only FP32 scorers and a small, reproducible C3 pilot trainer.\n\nNothing trains or downloads on import. Torch must already be installed in the\ncloud environment. NumPy train arrays stay on the host; only one sampled batch\nis transferred to the model device. ``dim`` counts REAL scalars per entity.\n\nInput queries contain scalar direction (0=head, 1=tail), relation[B], fixed[B],\nand optional bookkeeping fields. A scorer never receives/reads the target label.\nOutput scores[B,K] are float32, with larger meaning more plausible.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport platform\nimport random\nimport time\nfrom typing import Any, Callable, Mapping\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\ndef _canonical(value: Any) -> str:\n    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"),\n                      allow_nan=False)\n\n\ndef _sha(value: Any) -> str:\n    return hashlib.sha256(_canonical(value).encode("utf-8")).hexdigest()\n\n\ndef _utc() -> str:\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef _direction(query: Mapping[str, Any]) -> int:\n    value = query["direction"]\n    if not isinstance(value, (int, np.integer)) or isinstance(value, (bool, np.bool_)) or int(value) not in (0, 1):\n        raise ValueError("direction must be scalar 0=head or 1=tail")\n    return int(value)\n\n\nclass LearnedScorer(nn.Module):\n    """Common candidate scorer; row_id, target_type and split are not features."""\n\n    model_name: str\n\n    def __init__(self, n_entities: int, n_relations: int, dim: int, seed: int):\n        super().__init__()\n        self.n_entities = int(n_entities)\n        self.n_relations = int(n_relations)\n        self.dim = int(dim)\n        self.seed = int(seed)\n\n    @property\n    def device(self) -> torch.device:\n        return next(self.parameters()).device\n\n    def _index(self, values: Any) -> torch.Tensor:\n        return torch.as_tensor(values, dtype=torch.long, device=self.device)\n\n    def score_torch(self, query: Mapping[str, Any], candidates: Any) -> torch.Tensor:\n        raise NotImplementedError\n\n    def score_candidates(self, query: Mapping[str, Any], candidates: np.ndarray) -> np.ndarray:\n        """Evaluate one candidate chunk. The evaluator controls B/K for memory."""\n        previous_mode = self.training\n        self.eval()\n        try:\n            with torch.no_grad():\n                return self.score_torch(query, candidates).detach().cpu().numpy()\n        finally:\n            self.train(previous_mode)\n\n    def model_config(self) -> dict[str, Any]:\n        return {"name": self.model_name, "dim": self.dim, "seed": self.seed,\n                "n_entities": self.n_entities, "n_relations": self.n_relations,\n                "precision": "float32", "gamma": 12.0 if self.model_name == "transe" else None}\n\n\nclass HeadBlind(LearnedScorer):\n    model_name = "head_blind"\n\n    def __init__(self, n_entities: int, n_relations: int, dim: int, seed: int = 101):\n        super().__init__(n_entities, n_relations, dim, seed)\n        self.entity = nn.Embedding(n_entities, dim)\n        self.relation_role = nn.Parameter(torch.empty(n_relations, 2, dim))\n        self.bias = nn.Parameter(torch.zeros(n_entities, 2))\n        nn.init.normal_(self.entity.weight, std=1.0 / math.sqrt(dim))\n        nn.init.normal_(self.relation_role, std=1.0 / math.sqrt(dim))\n\n    def score_torch(self, query: Mapping[str, Any], candidates: Any) -> torch.Tensor:\n        direction = _direction(query)\n        relation = self._index(query["relation"])\n        candidate = self._index(candidates)\n        role = self.relation_role[relation, direction]\n        # Deliberately do not even read query[\'fixed\'].\n        return (self.entity(candidate) * role[:, None, :]).sum(-1) + self.bias[candidate, direction]\n\n\nclass TransE(LearnedScorer):\n    model_name = "transe"\n    gamma = 12.0\n\n    def __init__(self, n_entities: int, n_relations: int, dim: int, seed: int = 101):\n        super().__init__(n_entities, n_relations, dim, seed)\n        self.entity = nn.Embedding(n_entities, dim)\n        self.relation = nn.Embedding(n_relations, dim)\n        bound = (self.gamma + 2.0) / dim\n        nn.init.uniform_(self.entity.weight, -bound, bound)\n        nn.init.uniform_(self.relation.weight, -bound, bound)\n\n    def score_torch(self, query: Mapping[str, Any], candidates: Any) -> torch.Tensor:\n        direction = _direction(query)\n        fixed = self.entity(self._index(query["fixed"]))[:, None, :]\n        relation = self.relation(self._index(query["relation"]))[:, None, :]\n        candidate = self.entity(self._index(candidates))\n        displacement = fixed + relation - candidate if direction == 1 else candidate + relation - fixed\n        return self.gamma - displacement.abs().sum(-1)\n\n\nclass ComplEx(LearnedScorer):\n    model_name = "complex"\n\n    def __init__(self, n_entities: int, n_relations: int, dim: int, seed: int = 101):\n        if dim % 2:\n            raise ValueError("ComplEx dim counts real scalars and must be even")\n        super().__init__(n_entities, n_relations, dim, seed)\n        self.entity = nn.Embedding(n_entities, dim)\n        self.relation = nn.Embedding(n_relations, dim)\n        nn.init.normal_(self.entity.weight, std=1.0 / math.sqrt(dim // 2))\n        nn.init.normal_(self.relation.weight, std=1.0 / math.sqrt(dim // 2))\n\n    def score_torch(self, query: Mapping[str, Any], candidates: Any) -> torch.Tensor:\n        direction = _direction(query)\n        fixed = self.entity(self._index(query["fixed"]))[:, None, :]\n        candidate = self.entity(self._index(candidates))\n        relation = self.relation(self._index(query["relation"]))[:, None, :]\n        head, tail = (fixed, candidate) if direction == 1 else (candidate, fixed)\n        hr, hi = head.chunk(2, -1)\n        rr, ri = relation.chunk(2, -1)\n        tr, ti = tail.chunk(2, -1)\n        return (hr * rr * tr + hi * rr * ti + hr * ri * ti - hi * ri * tr).sum(-1)\n\n\ndef make_model(name: str, data: Mapping[str, Any], dim: int = 128, seed: int = 101,\n               device: str = "cuda") -> LearnedScorer:\n    """Initialize in an isolated RNG scope, then move only weights to device."""\n    if dim <= 0:\n        raise ValueError("dim must be positive")\n    requested = torch.device(device)\n    if requested.type == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("CUDA GPU unavailable. Enable a Kaggle GPU; CPU requires device=\'cpu\' explicitly.")\n    classes = {"head_blind": HeadBlind, "transe": TransE, "complex": ComplEx}\n    if name not in classes:\n        raise ValueError(f"Unknown learned model: {name!r}")\n    with torch.random.fork_rng(devices=[]):\n        # Seed CPU initialization only; do not disturb any existing CUDA stream.\n        torch.random.default_generator.manual_seed(seed)\n        model = classes[name](int(data["n_entities"]), int(data["n_relations"]), dim, seed)\n    return model.to(device=requested, dtype=torch.float32)\n\n\n@dataclass(frozen=True)\nclass TrainConfig:\n    """Short pilot defaults, intentionally smaller than SPEC final-run settings."""\n\n    seed: int = 101\n    steps: int = 500\n    batch_size: int = 256\n    negatives: int = 64\n    learning_rate: float = 0.001\n    log_every: int = 50\n    checkpoint_every: int = 250\n    max_minutes: float = 20.0\n    validation_every: int = 250\n    selection_max_per_relation: int | None = 20\n    selection_seed: int = 71\n\n    def validate(self) -> None:\n        for key in ("steps", "batch_size", "negatives", "log_every", "checkpoint_every",\n                    "validation_every"):\n            value = getattr(self, key)\n            if not isinstance(value, int) or isinstance(value, bool) or value <= 0:\n                raise ValueError(f"{key} must be a positive integer")\n        selection = self.selection_max_per_relation\n        if selection is not None and (not isinstance(selection, int) or isinstance(selection, bool) or selection <= 0):\n            raise ValueError("selection_max_per_relation must be a positive integer or None")\n        if not isinstance(self.seed, int) or self.seed < 0:\n            raise ValueError("seed must be a nonnegative integer")\n        if not math.isfinite(self.learning_rate) or self.learning_rate <= 0:\n            raise ValueError("learning_rate must be finite and positive")\n        if not math.isfinite(self.max_minutes) or self.max_minutes <= 0:\n            raise ValueError("max_minutes must be finite and positive")\n\n    def scientific(self) -> dict[str, Any]:\n        value = asdict(self)\n        # These affect only artifact frequency / this invocation\'s walltime cap.\n        for key in ("log_every", "checkpoint_every", "max_minutes"):\n            value.pop(key)\n        return value\n\n\ndef _identity(data: Mapping[str, Any]) -> dict[str, str]:\n    """Use the prepared manifest, never rehash multi-GB arrays every checkpoint.\n\n    The loader/audit must verify the file hashes declared in this manifest.\n    Manifest identity alone cannot detect arrays altered after that verification.\n    """\n    return {"mapping_sha256": _sha(data["mapping"]),\n            "dataset_manifest_sha256": _sha(data["manifest"])}\n\n\ndef _environment(model: LearnedScorer) -> dict[str, Any]:\n    device = model.device\n    return {"python": platform.python_version(), "platform": platform.platform(),\n            "numpy": np.__version__, "torch": str(torch.__version__),\n            "cuda_runtime": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),\n            "device": str(device),\n            "gpu_name": torch.cuda.get_device_name(device) if device.type == "cuda" else None,\n            "cuda_device_count": torch.cuda.device_count() if device.type == "cuda" else 0,\n            "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),\n            "cudnn_deterministic": torch.backends.cudnn.deterministic,\n            "cudnn_benchmark": torch.backends.cudnn.benchmark,\n            "cuda_matmul_tf32": torch.backends.cuda.matmul.allow_tf32,\n            "cudnn_tf32": torch.backends.cudnn.allow_tf32}\n\n\ndef _source_sha() -> str:\n    return hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\n\n\ndef _sync(model: LearnedScorer) -> None:\n    if model.device.type == "cuda":\n        torch.cuda.synchronize(model.device)\n\n\ndef _json_new(path: Path, value: Mapping[str, Any]) -> None:\n    with path.open("x", encoding="utf-8") as handle:\n        handle.write(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\\n")\n\n\ndef _unused_path(directory: Path, stem: str, extension: str) -> Path:\n    candidate = directory / f"{stem}{extension}"\n    suffix = 1\n    while candidate.exists():\n        candidate = directory / f"{stem}_{suffix:03d}{extension}"\n        suffix += 1\n    return candidate\n\n\ndef _streams(seed: int) -> tuple[np.random.Generator, np.random.Generator]:\n    # Explicit independent streams; model initialization cannot advance either.\n    return (np.random.default_rng(np.random.SeedSequence([seed, 1701])),\n            np.random.default_rng(np.random.SeedSequence([seed, 1702])))\n\n\ndef _sample_batch(data: Mapping[str, Any], config: TrainConfig, step: int,\n                  positive_rng: np.random.Generator,\n                  negative_rng: np.random.Generator) -> tuple[dict[str, Any], np.ndarray, str]:\n    train = data["train"]\n    rows = positive_rng.integers(0, len(train["head"]), size=config.batch_size, dtype=np.int64)\n    direction = 1 if step % 2 == 1 else 0  # update 1 tail, update 2 head\n    target_name, fixed_name = ("tail", "head") if direction else ("head", "tail")\n    types = np.asarray(train[f"{target_name}_type"][rows], dtype=np.int64)\n    offsets = np.asarray(data["type_offsets"], dtype=np.int64)[types]\n    sizes = np.asarray(data["type_sizes"], dtype=np.int64)[types]\n    # Integer RNG accepts per-row upper bounds; duplicates/positives are allowed.\n    negative = negative_rng.integers(0, sizes[:, None],\n                                     size=(config.batch_size, config.negatives), dtype=np.int64)\n    negative += offsets[:, None]\n    positive = np.asarray(train[target_name][rows], dtype=np.int64)\n    candidates = np.concatenate((positive[:, None], negative), axis=1)\n    query = {"row_id": rows, "direction": direction,\n             "relation": np.asarray(train["relation"][rows], dtype=np.int64),\n             "fixed": np.asarray(train[fixed_name][rows], dtype=np.int64), "target_type": types}\n    digest = hashlib.sha256()\n    digest.update(bytes([direction]))\n    for values in (rows, query["relation"], query["fixed"], types, candidates):\n        digest.update(np.asarray(values, dtype="<i8").tobytes(order="C"))\n    return query, candidates, digest.hexdigest()\n\n\ndef _validate_train(data: Mapping[str, Any], model: LearnedScorer) -> None:\n    train = data["train"]\n    length = len(train["head"])\n    if length == 0:\n        raise ValueError("Train split is empty")\n    for name in ("head", "relation", "tail", "head_type", "tail_type"):\n        array = train[name]\n        if array.shape != (length,) or not np.issubdtype(array.dtype, np.integer):\n            raise ValueError(f"train[{name}] must be an integer vector with common length")\n    if (int(data["n_entities"]), int(data["n_relations"])) != (model.n_entities, model.n_relations):\n        raise ValueError("Model/data dimensions differ")\n    sizes = np.asarray(data["type_sizes"], dtype=np.int64)\n    offsets = np.asarray(data["type_offsets"], dtype=np.int64)\n    if sizes.ndim != 1 or offsets.shape != sizes.shape or np.any(sizes <= 0):\n        raise ValueError("Invalid typed entity universe")\n    if not np.array_equal(offsets, np.r_[0, np.cumsum(sizes)[:-1]]) or sizes.sum() != model.n_entities:\n        raise ValueError("Type offsets must partition the global entity universe")\n    for name, limit in (("head", model.n_entities), ("tail", model.n_entities),\n                        ("relation", model.n_relations), ("head_type", len(sizes)), ("tail_type", len(sizes))):\n        if int(np.min(train[name])) < 0 or int(np.max(train[name])) >= limit:\n            raise ValueError(f"train[{name}] is outside its ID universe")\n    # Chunk checks avoid allocating full-length temporary typed-ID arrays.\n    for start in range(0, length, 100_000):\n        stop = min(start + 100_000, length)\n        relation = np.asarray(train["relation"][start:stop], dtype=np.int64)\n        for role in ("head", "tail"):\n            entity = train[role][start:stop]\n            types = np.asarray(train[f"{role}_type"][start:stop], dtype=np.int64)\n            schema = np.asarray(data[f"schema_{role}"], dtype=np.int64)\n            if schema.shape != (model.n_relations,) or not np.array_equal(types, schema[relation]):\n                raise ValueError(f"Train {role} types disagree with relation schema")\n            if np.any(entity < offsets[types]) or np.any(entity >= offsets[types] + sizes[types]):\n                raise ValueError(f"Train {role} global IDs disagree with their types")\n\n\ndef _read_checkpoint(path: Path) -> dict[str, Any]:\n    """Load ONLY trusted checkpoints produced by this project (pickle payload)."""\n    return torch.load(path, map_location="cpu", weights_only=False)\n\n\ndef load_model_checkpoint(model: LearnedScorer, path: str | Path,\n                          data: Mapping[str, Any] | None = None) -> dict[str, Any]:\n    """Restore one saved scoring snapshot, not optimizer/training streams.\n\n    Return metadata so tables can label the actual saved step and selection kind.\n    The caller must supply a trusted project-created checkpoint.\n    """\n    checkpoint = _read_checkpoint(Path(path))\n    if checkpoint["format_version"] != 1 or checkpoint["model_config"] != model.model_config():\n        raise ValueError("Checkpoint/model configuration mismatch")\n    if data is not None and checkpoint["data_identity"] != _identity(data):\n        raise ValueError("Checkpoint dataset/mapping mismatch")\n    model.load_state_dict(checkpoint["model_state"], strict=True)\n    model.eval()\n    return {"step": checkpoint["step"], "kind": checkpoint["kind"],\n            "model_config": checkpoint["model_config"], "best_mrr": checkpoint["best_mrr"],\n            "created_at_utc": checkpoint["created_at_utc"]}\n\n\ndef train_model(model: LearnedScorer, data: Mapping[str, Any], config: TrainConfig,\n                run_dir: Path, evaluate_fn: Callable[[LearnedScorer, int], Mapping[str, Any]] | None = None,\n                resume_path: str | Path | None = None) -> dict[str, Any]:\n    """Train uniform typed batches and write immutable pilot artifacts.\n\n    evaluate_fn(model, step) must use one fixed train-independent validation ID\n    selection and return {\'mrr\': finite float, ...JSON metadata...}. Higher MRR\n    selects the checkpoint; ties retain the earlier checkpoint. No test is read.\n    max_minutes is a per-invocation cap checked between updates; a single update,\n    validation or checkpoint write may exceed the remaining time. It is not a\n    full final-run benchmark. Resume requires identical scientific config, source,\n    data identities and recorded device/software environment.\n    """\n    config.validate()\n    if config.seed != model.seed:\n        raise ValueError("Training seed must equal model initialization seed")\n    if any(parameter.dtype != torch.float32 for parameter in model.parameters()):\n        raise ValueError("Only float32 pilot models are supported")\n    _validate_train(data, model)\n    run_dir = Path(run_dir).resolve()\n    identity, environment = _identity(data), _environment(model)\n    scientific_sha = _sha(config.scientific())\n    source_sha = _source_sha()\n    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate,\n                                 betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0)\n    positive_rng, negative_rng = _streams(config.seed)\n    step = 0\n    best_mrr: float | None = None\n    best_checkpoint: str | None = None\n    last_checkpoint: str | None = None\n    validation_history: list[dict[str, Any]] = []\n    batch_hashes: list[dict[str, Any]] = []\n    timings: list[float] = []\n    previous_wall_seconds = 0.0\n    validation_enabled = evaluate_fn is not None\n    checkpoint = None\n    if resume_path is not None:\n        resume = Path(resume_path).resolve()\n        # An explicit resume may append only within the same immutable run.\n        if resume.parent != run_dir / "checkpoints" or not run_dir.is_dir():\n            raise ValueError("Resume checkpoint must belong to run_dir/checkpoints")\n        checkpoint = _read_checkpoint(resume)\n        expected = {"format_version": 1, "scientific_config_sha256": scientific_sha,\n                    "model_config": model.model_config(), "data_identity": identity,\n                    "environment": environment, "source_sha256": source_sha,\n                    "validation_enabled": validation_enabled, "run_dir": str(run_dir)}\n        for key, value in expected.items():\n            if checkpoint.get(key) != value:\n                raise ValueError(f"Strict resume mismatch: {key}")\n        run_metadata = json.loads((run_dir / "run.json").read_text(encoding="utf-8"))\n        for key in ("scientific_config_sha256", "model_config", "data_identity", "source_sha256",\n                    "validation_enabled"):\n            if run_metadata.get(key) != expected[key]:\n                raise ValueError(f"Run metadata mismatch: {key}")\n        step = int(checkpoint["step"])\n        if not 0 <= step <= config.steps:\n            raise ValueError("Checkpoint step outside requested run")\n        # Appending a branch from an older step would create contradictory logs.\n        history_path = run_dir / "history.jsonl"\n        if history_path.exists():\n            with history_path.open(encoding="utf-8") as handle:\n                for line in handle:\n                    event = json.loads(line)\n                    if int(event.get("step", 0)) > step:\n                        raise ValueError("Run has later logged steps; resume its latest checkpoint or use a separate run")\n        model.load_state_dict(checkpoint["model_state"], strict=True)\n        optimizer.load_state_dict(checkpoint["optimizer_state"])\n        positive_rng.bit_generator.state = checkpoint["positive_rng"]\n        negative_rng.bit_generator.state = checkpoint["negative_rng"]\n        random.setstate(checkpoint["python_rng"])\n        np.random.set_state(checkpoint["numpy_global_rng"])\n        torch.set_rng_state(checkpoint["torch_cpu_rng"])\n        if model.device.type == "cuda":\n            torch.cuda.set_rng_state_all(checkpoint["torch_cuda_rng"])\n        best_mrr = checkpoint["best_mrr"]\n        best_checkpoint = checkpoint["best_checkpoint"]\n        last_checkpoint = str(resume)\n        validation_history = checkpoint["validation_history"]\n        batch_hashes = checkpoint["first_batch_hashes"]\n        timings = checkpoint["step_seconds"]\n        previous_wall_seconds = checkpoint["total_wall_seconds"]\n    else:\n        if run_dir.exists():\n            raise FileExistsError("run_dir already exists; choose a new run ID or explicit resume")\n        run_dir.mkdir(parents=True, exist_ok=False)\n        (run_dir / "checkpoints").mkdir()\n        _json_new(run_dir / "run.json", {\n            "created_at_utc": _utc(), "run_dir": str(run_dir),\n            "config": asdict(config), "scientific_config_sha256": scientific_sha,\n            "model_config": model.model_config(), "data_identity": identity,\n            "environment": environment, "source_sha256": source_sha,\n            "validation_enabled": validation_enabled, "scope": "preliminary_validation_pilot",\n            "sampler": "uniform_train_rows; alternate_tail_head; typed_with_replacement; no_filter",\n            "loss": "balanced_logistic", "selection": "max_validation_micro_MRR; earlier_step_on_tie"})\n\n    started = time.perf_counter()\n    history = (run_dir / "history.jsonl").open("a", encoding="utf-8")\n\n    def log(event: Mapping[str, Any]) -> None:\n        history.write(_canonical({"utc": _utc(), **event}) + "\\n")\n        history.flush()\n\n    def save(kind: str) -> str:\n        nonlocal last_checkpoint, best_checkpoint\n        path = _unused_path(run_dir / "checkpoints", f"{kind}_step_{step:08d}", ".pt")\n        if kind == "best":\n            best_checkpoint = str(path)\n        else:\n            last_checkpoint = str(path)\n        payload = {\n            "format_version": 1, "created_at_utc": _utc(), "kind": kind, "step": step,\n            "run_dir": str(run_dir), "model_config": model.model_config(),\n            "config": asdict(config), "config_sha256": _sha(asdict(config)),\n            "scientific_config_sha256": scientific_sha, "source_sha256": source_sha,\n            "data_identity": identity, "environment": environment,\n            "validation_enabled": validation_enabled,\n            "model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),\n            "positive_rng": positive_rng.bit_generator.state,\n            "negative_rng": negative_rng.bit_generator.state,\n            "python_rng": random.getstate(), "numpy_global_rng": np.random.get_state(),\n            "torch_cpu_rng": torch.get_rng_state(),\n            "torch_cuda_rng": torch.cuda.get_rng_state_all() if model.device.type == "cuda" else [],\n            "best_mrr": best_mrr, "best_checkpoint": best_checkpoint,\n            "last_checkpoint": last_checkpoint, "validation_history": validation_history,\n            "first_batch_hashes": batch_hashes, "step_seconds": timings,\n            "total_wall_seconds": previous_wall_seconds + time.perf_counter() - started}\n        with path.open("xb") as handle:\n            torch.save(payload, handle)\n        log({"event": "checkpoint", "step": step, "kind": kind, "path": str(path)})\n        return str(path)\n\n    stop_reason = "completed_requested_steps"\n    try:\n        log({"event": "resume" if checkpoint is not None else "start", "step": step,\n             "runtime_config": asdict(config)})\n        model.train()\n        while step < config.steps:\n            if time.perf_counter() - started >= config.max_minutes * 60:\n                stop_reason = "wall_time_cap"\n                break\n            next_step = step + 1\n            query, candidates, fingerprint = _sample_batch(data, config, next_step, positive_rng, negative_rng)\n            _sync(model)\n            began = time.perf_counter()\n            optimizer.zero_grad(set_to_none=True)\n            scores = model.score_torch(query, candidates)\n            loss = -0.5 * F.logsigmoid(scores[:, 0]).mean() - 0.5 * F.logsigmoid(-scores[:, 1:]).mean()\n            if not bool(torch.isfinite(loss).item()):\n                # Do not save an unusable resume stream after sampling a failed batch.\n                raise FloatingPointError(f"Nonfinite loss at update {next_step}; use the previous checkpoint")\n            loss.backward()\n            if any(parameter.grad is not None and not bool(torch.isfinite(parameter.grad).all().item())\n                   for parameter in model.parameters()):\n                raise FloatingPointError(f"Nonfinite gradient at update {next_step}; use the previous checkpoint")\n            optimizer.step()\n            _sync(model)\n            timings.append(time.perf_counter() - began)\n            step = next_step\n            if len(batch_hashes) < 10:\n                batch_hashes.append({"step": step, "sha256": fingerprint, "direction": query["direction"]})\n                log({"event": "batch_fingerprint", **batch_hashes[-1]})\n            if step % config.log_every == 0 or step == 1 or step == config.steps:\n                log({"event": "update", "step": step, "loss": float(loss.detach().cpu()),\n                     "step_seconds": timings[-1]})\n            if evaluate_fn is not None and (step % config.validation_every == 0 or step == config.steps):\n                model.eval()\n                try:\n                    with torch.no_grad():\n                        metrics = dict(evaluate_fn(model, step))\n                finally:\n                    model.train()\n                mrr = float(metrics["mrr"])\n                if not math.isfinite(mrr) or not 0 <= mrr <= 1:\n                    raise ValueError("Validation mrr must be finite and in [0,1]")\n                record = {"step": step, **metrics}\n                _canonical(record)  # Reject non-JSON/nonfinite callback metadata before save.\n                validation_history.append(record)\n                log({"event": "validation", **record})\n                if best_mrr is None or mrr > best_mrr:\n                    best_mrr = mrr\n                    save("best")\n            if step % config.checkpoint_every == 0:\n                save("last")\n        # Keep the current snapshot even if the wall-time cap interrupted the pilot.\n        save("last")\n        elapsed = time.perf_counter() - started\n        result = {\n            "scope": "preliminary_validation_pilot", "completed_steps": step,\n            "requested_steps": config.steps, "completed": step == config.steps,\n            "stop_reason": stop_reason, "best_checkpoint": best_checkpoint,\n            "last_checkpoint": last_checkpoint, "best_mrr": best_mrr,\n            "validation_history": validation_history, "first_batch_hashes": batch_hashes,\n            "step_sec": float(np.median(timings)) if timings else None,\n            "step_sec_p90": float(np.quantile(timings, 0.9)) if timings else None,\n            "measured_update_count": len(timings),\n            "invocation_wall_seconds": elapsed, "total_wall_seconds": previous_wall_seconds + elapsed,\n            "model_config": model.model_config(), "config": asdict(config),\n            "data_identity": identity, "environment": environment,\n            "note": "Pilot timing includes gradient-finite checks; no warmup exclusion; not a final benchmark."}\n        manifest_path = _unused_path(run_dir, f"result_step_{step:08d}", ".json")\n        result["result_manifest"] = str(manifest_path)\n        _json_new(manifest_path, result)\n        log({"event": "stop", "step": step, "reason": stop_reason, "result_manifest": str(manifest_path)})\n        return result\n    finally:\n        history.close()\n\n\ndef model_gates() -> dict[str, Any]:\n    """Future synthetic CPU gate; callable explicitly, never executed on import.\n\n    Compares real-expanded ComplEx against direct complex arithmetic in BOTH\n    directions, TransE against L1, and HeadBlind against fixed-ID permutation.\n    These are synthetic checks, not ogbl-biokg measurements.\n    """\n    fixture = {"n_entities": 7, "n_relations": 3}\n    candidates = np.array([[0, 2, 2], [1, 5, 6]], dtype=np.int64)\n    query = {"relation": np.array([0, 2]), "fixed": np.array([3, 4]), "direction": 1}\n    passed: dict[str, Any] = {"scope": "synthetic_model_gates"}\n    for name in ("transe", "complex", "head_blind"):\n        model = make_model(name, fixture, dim=8, seed=101, device="cpu")\n        with torch.no_grad():\n            for direction in (0, 1):\n                q = {**query, "direction": direction}\n                actual = model.score_torch(q, candidates)\n                if name == "transe":\n                    fixed = model.entity(model._index(q["fixed"]))[:, None, :]\n                    entity = model.entity(model._index(candidates))\n                    relation = model.relation(model._index(q["relation"]))[:, None, :]\n                    head, tail = (fixed, entity) if direction else (entity, fixed)\n                    expected = 12.0 - torch.linalg.vector_norm(head + relation - tail, ord=1, dim=-1)\n                elif name == "complex":\n                    def as_complex(tensor: torch.Tensor) -> torch.Tensor:\n                        real, imaginary = tensor.chunk(2, -1)\n                        return torch.complex(real, imaginary)\n                    fixed = as_complex(model.entity(model._index(q["fixed"])))[:, None, :]\n                    entity = as_complex(model.entity(model._index(candidates)))\n                    relation = as_complex(model.relation(model._index(q["relation"])))[:, None, :]\n                    head, tail = (fixed, entity) if direction else (entity, fixed)\n                    expected = (head * relation * torch.conj(tail)).sum(-1).real\n                else:\n                    changed = {**q, "fixed": np.array([6, 0])}\n                    expected = model.score_torch(changed, candidates)\n                torch.testing.assert_close(actual, expected, rtol=2e-5, atol=2e-6)\n                # Same IDs have same scores, regardless of column position/chunk.\n                permutation = np.array([2, 0, 1])\n                torch.testing.assert_close(model.score_torch(q, candidates[:, permutation]),\n                                           actual[:, permutation], rtol=0, atol=0)\n                chunks = torch.cat([model.score_torch(q, candidates[:, i:i + 1]) for i in range(3)], dim=1)\n                torch.testing.assert_close(actual, chunks, rtol=2e-5, atol=2e-6)\n            if name == "head_blind":\n                # Construct a deterministic role difference, rather than relying on chance.\n                model.relation_role[:, 0].zero_()\n                model.relation_role[:, 1].fill_(1)\n                model.entity.weight.fill_(1)\n                head = model.score_torch({**query, "direction": 0}, candidates)\n                tail = model.score_torch({**query, "direction": 1}, candidates)\n                if torch.equal(head, tail):\n                    raise AssertionError("HeadBlind must distinguish head and tail roles")\n        passed[name] = "passed"\n    return passed\n', 'workflow.py': '"""Small cloud workflow utilities; no work is performed at import time."""\nfrom __future__ import annotations\n\nimport csv\nfrom datetime import datetime, timezone\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\nfrom pathlib import Path\nimport shutil\nimport time\nimport zipfile\n\n\ndef environment_report() -> dict:\n    import torch\n    versions = {}\n    for package in (\'torch\', \'numpy\', \'ogb\', \'pandas\', \'scikit-learn\'):\n        try:\n            versions[package] = importlib.metadata.version(package)\n        except importlib.metadata.PackageNotFoundError:\n            versions[package] = None\n    result = dict(python=platform.python_version(), platform=platform.platform(),\n                  packages=versions, cuda_available=torch.cuda.is_available(),\n                  torch_cuda=torch.version.cuda, devices=[])\n    if torch.cuda.is_available():\n        for index in range(torch.cuda.device_count()):\n            properties = torch.cuda.get_device_properties(index)\n            result[\'devices\'].append(dict(index=index, name=properties.name,\n                                          total_memory_bytes=properties.total_memory))\n    return result\n\n\ndef new_run(workspace: Path, config: dict, stage: str) -> Path:\n    workspace = Path(workspace).resolve()\n    workspace.mkdir(parents=True, exist_ok=True)\n    usage = shutil.disk_usage(workspace)\n    digest = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:8]\n    stamp = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S\')\n    run = workspace/\'runs\'/f\'{stamp}_{time.time_ns()}_{stage}_{digest}\'\n    run.mkdir(parents=True, exist_ok=False)\n    save_json(run/\'config.json\', config)\n    save_json(run/\'environment.json\', environment_report())\n    save_json(run/\'disk_before.json\', dict(free_bytes=usage.free, total_bytes=usage.total))\n    return run\n\n\ndef save_json(path: Path, value) -> None:\n    with Path(path).open(\'x\', encoding=\'utf-8\') as handle:\n        json.dump(value, handle, ensure_ascii=False, indent=2)\n\n\ndef write_comparison(run_dir: Path, results: list[dict]) -> Path:\n    if not results:\n        raise ValueError(\'No evaluation results to compare\')\n    row_hashes = {x[\'validation_rows_sha256\'] for x in results}\n    candidate_hashes = {x[\'candidate_sha256\'] for x in results}\n    if len(row_hashes) != 1 or len(candidate_hashes) != 1:\n        raise ValueError(\'Scorers used different questions or candidates\')\n    path = Path(run_dir)/f\'comparison_{time.time_ns()}.csv\'\n    columns = [\'model\', \'training_seed\', \'n_triples\', \'n_queries\', \'mrr\',\n               \'hits1\', \'hits3\', \'hits10\', \'macro_relation_role_mrr\',\n               \'evaluation_seconds\', \'stage\', \'split\']\n    with path.open(\'x\', encoding=\'utf-8\', newline=\'\') as handle:\n        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction=\'ignore\')\n        writer.writeheader()\n        writer.writerows(results)\n    return path\n\n\ndef export_run(run_dir: Path) -> Path:\n    """Zip just this run, retaining all originals. No dataset or private paths."""\n    run_dir = Path(run_dir).resolve()\n    archive = run_dir.parent/f\'{run_dir.name}_export_{time.time_ns()}.zip\'\n    with zipfile.ZipFile(archive, mode=\'x\', compression=zipfile.ZIP_DEFLATED,\n                         compresslevel=1) as target:\n        # All descendants were produced by this project. Do not follow links.\n        def add_directory(folder):\n            for path in sorted(folder.iterdir()):\n                if path.is_symlink() or getattr(path, \'is_junction\', lambda: False)():\n                    raise ValueError(f\'Linked path in run output: {path.name}\')\n                if not path.resolve().is_relative_to(run_dir):\n                    raise ValueError(\'Output escapes run directory\')\n                if path.is_dir():\n                    add_directory(path)\n                elif path.is_file():\n                    target.write(path, path.relative_to(run_dir.parent))\n        add_directory(run_dir)\n    return archive\n'}

SOURCE_HASHES = {'__init__.py': '98e7fc2df87a9266d164dd0e7197e9dab51487a922015893f65614af7e08c1f7', 'data.py': 'ea5c1b229e05f2f6afc7bd662c7273d473281874ffceee0cdefc9f1793923f30', 'baselines.py': 'b502e82ff20186bbd64b4d685357a12147bcffcfac5214e152d50d96d83c657a', 'evaluate.py': '658fd74b5c70e715f7536a335b93d40ccccfb47c7e2dd5b2a403d2d87c371b88', 'training.py': '07ff2003e27ede98ca5ba84e1e9683dd531e3fc71e23cf08ee755dd09fe4430a', 'workflow.py': '8f548bae09175a2848499b24e54285fa4e1eae828980c2912df745bce69205db'}

if MODE != "prepare_only":
    import hashlib
    import sys
    PACKAGE_DIR = WORKSPACE / "source" / "biokg_audit"
    PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
    for name, source in SOURCE_BUNDLE.items():
        target = PACKAGE_DIR / name
        if target.is_symlink():
            raise RuntimeError("Вместо файла кода обнаружена ссылка")
        if target.exists():
            actual = hashlib.sha256(target.read_bytes()).hexdigest()
            if actual != SOURCE_HASHES[name]:
                raise RuntimeError(f"Старый файл отличается: {name}. Выбери новую WORKSPACE; ничего не удаляй.")
        else:
            with target.open("x", encoding="utf-8", newline="") as handle:
                handle.write(source)
    sys.path.insert(0, str(PACKAGE_DIR.parent))
    # В новой сессии импортируется именно этот пакет. При изменении исходников
    # в уже работающем kernel следует начать новую сессию, не использовать reload.
    print("Исходники подготовлены:", PACKAGE_DIR)
else:
    print("prepare_only: файлы cloud-пакета не создавались.")


Исходники подготовлены: /kaggle/working/dkr_c3/source/biokg_audit


## 3. Проверка окружения

Для baseline GPU не нужен. Для pilot эта ячейка проверяет реальную CUDA-операцию
на одной карте и печатает название устройства. Если Torch не видит GPU, обучение
остановится: переключение на медленный CPU не произойдёт незаметно.


In [7]:
if MODE != "prepare_only":
    import importlib.util
    import importlib.metadata
    import subprocess
    import sys
    import json
    for module in ("torch", "numpy", "pandas", "sklearn", "tqdm", "six", "urllib3"):
        if importlib.util.find_spec(module) is None:
            raise RuntimeError(f"В Kaggle отсутствует {module}. Не переустанавливай Torch вслепую.")
    try:
        ogb_version = importlib.metadata.version("ogb")
    except importlib.metadata.PackageNotFoundError:
        ogb_version = None
    if ogb_version != "1.3.6":
        if not ALLOW_OGB_INSTALL:
            raise RuntimeError("Для официальной проверки нужен ogb==1.3.6. Разреши ALLOW_OGB_INSTALL или подготовь wheel как Input.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "ogb==1.3.6"], check=True)
    import torch
    from biokg_audit.workflow import environment_report
    ENVIRONMENT = environment_report()
    print(json.dumps(ENVIRONMENT, ensure_ascii=False, indent=2))
    if MODE == "pilot":
        if not torch.cuda.is_available():
            raise RuntimeError("Включи GPU T4 в настройках Kaggle перед pilot.")
        torch.ones(4, device="cuda:0").sum().item()
        torch.cuda.synchronize(0)
        print("CUDA-операция выполнена. Используем только cuda:0.")


{
  "python": "3.13.16",
  "platform": "Linux-6.18.55+-x86_64-with-glibc2.39",
  "packages": {
    "torch": "2.11.0+cu130",
    "numpy": "2.1.3",
    "ogb": "1.3.6",
    "pandas": "2.3.3",
    "scikit-learn": "1.6.1"
  },
  "cuda_available": true,
  "torch_cuda": "13.0",
  "devices": [
    {
      "index": 0,
      "name": "Tesla T4",
      "total_memory_bytes": 15636037632
    },
    {
      "index": 1,
      "name": "Tesla T4",
      "total_memory_bytes": 15636037632
    }
  ]
}


## 4. Проверяем, правильно ли считаются баллы

Это маленькие искусственные примеры. Наш rank сравнивается с установленным
официальным evaluator OGB, включая одинаковые scores. Если проверка не прошла,
дальше запускаться нельзя. Это ещё не результат на биомедицинском датасете.


In [9]:
# Replace ONLY the failed metric-check cell with this code in the current Kaggle
# notebook. No restart, reinstall, data download or training is performed here.
import hashlib
from pathlib import Path
import numpy as np
import torch
import ogb
import ogb.linkproppred.evaluate as official_module
from ogb.linkproppred import Evaluator
from biokg_audit.evaluate import midrank_metrics

METRIC_REPORT = None
rng = np.random.default_rng(71)
positive = np.r_[1.0, -1.0, 0.0, 0.0, rng.normal(size=28)]
negative = np.vstack([np.zeros((4, 500)), rng.normal(size=(28, 500))])
negative[3] = -1.0
negative[3, :2] = 1.0
negative[3, 2:6] = 0.0
ours = midrank_metrics(positive, negative)
reference = Evaluator(name="ogbl-biokg").eval({
    "y_pred_pos": torch.from_numpy(positive),
    "y_pred_neg": torch.from_numpy(negative),
})
for local, official in [("rr", "mrr_list"), ("hits1", "hits@1_list"),
                        ("hits3", "hits@3_list"), ("hits10", "hits@10_list")]:
    np.testing.assert_allclose(ours[local], reference[official].cpu().numpy(), atol=1e-7, rtol=1e-7)
np.testing.assert_equal(ours["rank"][:4], [1.0, 501.0, 251.0, 5.0])
np.testing.assert_array_equal(ours["rank"], midrank_metrics(positive, negative[:, ::-1])["rank"])
METRIC_REPORT = {
    "status": "passed", "official_backend": "torch_cpu", "ogb_version": ogb.__version__,
    "official_evaluator_sha256": hashlib.sha256(Path(official_module.__file__).read_bytes()).hexdigest(),
    "checks": ["first_last_all_ties", "partial_ties", "numpy_vs_official_torch_cpu_parity", "candidate_permutation"],
}
print(METRIC_REPORT)


{'status': 'passed', 'official_backend': 'torch_cpu', 'ogb_version': '1.3.6', 'official_evaluator_sha256': 'cb2d2e7387c1d74043d77214cb2904508bac1e12ac331fa75415facaae40b3f4', 'checks': ['first_last_all_ties', 'partial_ties', 'numpy_vs_official_torch_cpu_parity', 'candidate_permutation']}


## 5. Загружаем данные и проверяем их

Читаются только train и validation. Test не распаковывается и не передаётся модели.
Большие числовые массивы сохраняются в кеш и читаются с диска порциями.

Проверяются типы объектов, ID, размеры, прямое пересечение train/valid и
официальные validation-negatives против известных train/valid фактов.
Проверка против test пока не выполнена — это ограничение первого этапа.
Первый аудит может занять время на CPU. Если найдено противоречие, не обходи проверку.


In [11]:
DATA = STATS = RUN_DIR = AUDIT_REPORT = BASELINE_REPORT = ROWS = None
if MODE != "prepare_only":
    if not METRIC_REPORT or METRIC_REPORT.get("status") != "passed":
        raise RuntimeError("Сначала нужна успешная проверка официальных метрик в разделе 4.")
    from biokg_audit.data import load_data, audit_data
    from biokg_audit.baselines import prepare_statistics
    from biokg_audit.evaluate import select_validation_rows, baseline_gates
    from biokg_audit.workflow import new_run, save_json
    import numpy as np

    RUN_CONFIG = {
        "mode": MODE, "stage": "preliminary_validation_only",
        "max_valid_per_relation": MAX_VALID_PER_RELATION,
        "validation_seed": VALIDATION_SEED,
        "eval_batch_size": EVAL_BATCH, "candidate_chunk": CANDIDATE_CHUNK,
        "pilot_model": PILOT_MODEL, "pilot_dim": PILOT_DIM,
        "pilot_steps": PILOT_STEPS, "pilot_batch": PILOT_BATCH,
        "pilot_negatives": PILOT_NEGATIVES, "pilot_seed": PILOT_SEED,
        "pilot_lr": PILOT_LR, "pilot_max_minutes": PILOT_MAX_MINUTES,
        "source_hashes": SOURCE_HASHES,
    }
    RUN_DIR = new_run(WORKSPACE, RUN_CONFIG, MODE)
    save_json(RUN_DIR / "metric_gates.json", METRIC_REPORT)
    DATA = load_data(WORKSPACE, input_dir=INPUT_DIR, download=ALLOW_DOWNLOAD)
    save_json(RUN_DIR / "dataset_manifest.json", DATA["manifest"])
    save_json(RUN_DIR / "entity_mapping.json", DATA["mapping"])
    AUDIT_REPORT = audit_data(DATA, RUN_DIR / "audit")
    STATS = prepare_statistics(DATA)
    save_json(RUN_DIR / "train_frequency_cells.json", STATS["frequency_cells"])
    save_json(RUN_DIR / "train_reverse_diagnostics.json", STATS["reverse_diagnostics"])
    BASELINE_REPORT = baseline_gates(DATA, STATS)
    save_json(RUN_DIR / "baseline_gates.json", BASELINE_REPORT)
    ROWS = select_validation_rows(DATA, MAX_VALID_PER_RELATION, VALIDATION_SEED)
    with (RUN_DIR / "validation_rows.npy").open("xb") as handle:
        np.save(handle, ROWS, allow_pickle=False)
    print("Аудит:", AUDIT_REPORT["status"])
    print("Исходных validation-троек:", len(DATA["valid"]["head"]))
    print("Выбрано троек:", len(ROWS), "; вопросов с учётом head/tail:", 2 * len(ROWS))
    print("На каждый вопрос сохраняются все 500 официальных negatives.")
    print("Папка результата:", RUN_DIR)


Auditing train/valid IDs, overlap and official negatives on CPU.
Auditing head negatives (all 500 per validation row).
Auditing tail negatives (all 500 per validation row).
Аудит: passed_first_stage_only
Исходных validation-троек: 162886
Выбрано троек: 963 ; вопросов с учётом head/tail: 1926
На каждый вопрос сохраняются все 500 официальных negatives.
Папка результата: /kaggle/working/dkr_c3/runs/20261112T122115_1794486075115976605_baselines_6dc8c4bb


## 6. Три простых baseline

- **Random** расставляет кандидатов случайно, но воспроизводимо.
- **Count prior** выше ставит тех, кто чаще встречался в нужном отношении в train.
- **Reverse lookup** ищет обратное ребро в train; при равенстве использует частоту.

Им не нужно обучать нейросеть. Таблица покажет, сколько баллов даёт каждая стратегия
на **одних и тех же** вопросах. Это маленькая стратифицированная validation-выборка:
её micro-MRR нельзя выдавать за полный официальный test MRR.


In [12]:
RESULTS = []
COMPARISON_PATH = None
if MODE != "prepare_only":
    if not METRIC_REPORT or METRIC_REPORT.get("status") != "passed":
        raise RuntimeError("Сначала нужна успешная проверка метрик.")
    if not AUDIT_REPORT or not AUDIT_REPORT["required_first_stage_gates_passed"]:
        raise RuntimeError("Сначала нужен успешный аудит данных.")
    if not BASELINE_REPORT or BASELINE_REPORT.get("status") != "passed":
        raise RuntimeError("Сначала нужны успешные проверки baseline в разделе 5.")
    from biokg_audit.baselines import RandomScorer, CountPrior, ReverseLookup
    from biokg_audit.evaluate import evaluate_validation
    from biokg_audit.workflow import write_comparison
    import pandas as pd
    from IPython.display import display
    SCORERS = {"random_seed101": RandomScorer(101),
               "count_prior": CountPrior(STATS), "reverse_lookup": ReverseLookup(STATS)}
    for name, scorer in SCORERS.items():
        result = evaluate_validation(
            scorer, DATA, STATS, ROWS, output_dir=RUN_DIR / name,
            model_name=name, batch_size=EVAL_BATCH, candidate_chunk=CANDIDATE_CHUNK)
        RESULTS.append(result)
    COMPARISON_PATH = write_comparison(RUN_DIR, RESULTS)
    display(pd.read_csv(COMPARISON_PATH))
    print("MRR: выше лучше; это не accuracy и не медицинская вероятность.")
    print("Подробные relation/frequency/reverse-срезы лежат в strata.csv каждого scorer.")


random_seed101: 0 / 64/963 triples
random_seed101: 0 / 963/963 triples
random_seed101: 1 / 64/963 triples
random_seed101: 1 / 963/963 triples
count_prior: 0 / 64/963 triples
count_prior: 0 / 963/963 triples
count_prior: 1 / 64/963 triples
count_prior: 1 / 963/963 triples
reverse_lookup: 0 / 64/963 triples
reverse_lookup: 0 / 963/963 triples
reverse_lookup: 1 / 64/963 triples
reverse_lookup: 1 / 963/963 triples


,model,training_seed,n_triples,n_queries,mrr,hits1,hits3,hits10,macro_relation_role_mrr,evaluation_seconds,stage,split
0,random_seed101,NaN,963,1926,0.016583,0.004673,0.008827,0.022845,0.016116,0.164466,preliminary_validation,valid
1,count_prior,NaN,963,1926,0.478156,0.321911,0.551921,0.809969,0.484781,0.139478,preliminary_validation,valid
2,reverse_lookup,NaN,963,1926,0.478156,0.321911,0.551921,0.809969,0.484781,0.323080,preliminary_validation,valid


MRR: выше лучше; это не accuracy и не медицинская вероятность.
Подробные relation/frequency/reverse-срезы лежат в strata.csv каждого scorer.


## 7. Необязательный короткий pilot на T4

Эта часть работает **только при MODE = "pilot"**. Для первого запуска выбран
HeadBlind: он учит рейтинг кандидатов, не видя известный другой объект запроса.
После рабочего пилота можно отдельно выбрать `complex` или `transe`.

По умолчанию: **одна модель, один seed, размерность 128, до 500 шагов**.
Ограничение 20 минут проверяется между шагами; одна оценка или запись checkpoint
может выйти за остаток времени. Это не обещание длительности полного проекта.
Если cap сработал, результат будет помечен незавершённым pilot.

Обучение использует только train; checkpoint выбирается на сохранённой validation-
выборке. Финальная таблица ниже использует те же ROWS и negatives, что baseline.


In [13]:
TRAIN_RESULT = CHECKPOINT_META = None
if MODE == "pilot":
    if not METRIC_REPORT or METRIC_REPORT.get("status") != "passed":
        raise RuntimeError("Обучение запрещено без успешной проверки метрик.")
    if not AUDIT_REPORT or not AUDIT_REPORT.get("required_first_stage_gates_passed"):
        raise RuntimeError("Обучение запрещено без успешного аудита данных.")
    if not BASELINE_REPORT or BASELINE_REPORT.get("status") != "passed":
        raise RuntimeError("Обучение запрещено без успешных baseline gates.")
    if DATA is None or STATS is None or ROWS is None or RUN_DIR is None:
        raise RuntimeError("Сначала выполни подготовку данных в разделе 5.")
    from biokg_audit.training import (make_model, TrainConfig, train_model,
                                     load_model_checkpoint, model_gates)
    from biokg_audit.workflow import save_json
    save_json(RUN_DIR / "synthetic_model_gates.json", model_gates())
    MODEL = make_model(PILOT_MODEL, DATA, dim=PILOT_DIM, seed=PILOT_SEED, device="cuda:0")
    CONFIG = TrainConfig(seed=PILOT_SEED, steps=PILOT_STEPS,
                         batch_size=PILOT_BATCH, negatives=PILOT_NEGATIVES,
                         learning_rate=PILOT_LR, max_minutes=PILOT_MAX_MINUTES,
                         checkpoint_every=250, validation_every=250,
                         selection_max_per_relation=MAX_VALID_PER_RELATION,
                         selection_seed=VALIDATION_SEED)
    # Exactly these row IDs are used by every checkpoint-selection callback.
    def validation_callback(model, step):
        result = evaluate_validation(model, DATA, STATS, ROWS,
                                     model_name=PILOT_MODEL, training_seed=PILOT_SEED,
                                     batch_size=EVAL_BATCH, candidate_chunk=CANDIDATE_CHUNK,
                                     save_queries=False, verbose=False)
        print("Validation step", step, "MRR", round(result["mrr"], 6))
        return result

    print("Параметров:", sum(p.numel() for p in MODEL.parameters()))
    torch.cuda.reset_peak_memory_stats(0)
    TRAIN_RESULT = train_model(MODEL, DATA, CONFIG, RUN_DIR / "training",
                               evaluate_fn=validation_callback)
    MEMORY_REPORT = {
        "peak_allocated_bytes": torch.cuda.max_memory_allocated(0),
        "peak_reserved_bytes": torch.cuda.max_memory_reserved(0),
        "step_sec_median": TRAIN_RESULT["step_sec"],
        "completed_steps": TRAIN_RESULT["completed_steps"],
        "stop_reason": TRAIN_RESULT["stop_reason"],
        "note": "Pilot measurements, not a full final-run resource guarantee.",
    }
    save_json(RUN_DIR / "pilot_memory_and_time.json", MEMORY_REPORT)
    CHOSEN_CHECKPOINT = TRAIN_RESULT["best_checkpoint"] or TRAIN_RESULT["last_checkpoint"]
    CHECKPOINT_META = load_model_checkpoint(MODEL, CHOSEN_CHECKPOINT, data=DATA)
    print("Остановка:", TRAIN_RESULT["stop_reason"], "; шагов:", TRAIN_RESULT["completed_steps"])
    print("Оцениваем checkpoint:", CHECKPOINT_META)
    learned = evaluate_validation(MODEL, DATA, STATS, ROWS,
                                  output_dir=RUN_DIR / "learned_validation",
                                  model_name=PILOT_MODEL, training_seed=PILOT_SEED,
                                  batch_size=EVAL_BATCH, candidate_chunk=CANDIDATE_CHUNK)
    learned["checkpoint_step"] = CHECKPOINT_META["step"]
    learned["checkpoint_kind"] = CHECKPOINT_META["kind"]
    learned["pilot_completed_requested_steps"] = TRAIN_RESULT["completed"]
    save_json(RUN_DIR / "learned_checkpoint_context.json", learned)
    RESULTS.append(learned)
    COMPARISON_PATH = write_comparison(RUN_DIR, RESULTS)
    display(pd.read_csv(COMPARISON_PATH))
elif MODE == "baselines":
    print("Обучение отключено: MODE=baselines. T4 для этого этапа не требуется.")


Обучение отключено: MODE=baselines. T4 для этого этапа не требуется.


## 8. Сохранить результат

В папке запуска уже есть config, среда, список вопросов, аудит и таблицы.
В pilot также сохраняются веса и история. Эта ячейка упакует **только результаты
текущего запуска**, не датасет. Исходные файлы остаются на месте.

Скачай ZIP из Kaggle Output до потери сессии. По `comparison*.csv`, `audit/` и
`pilot_memory_and_time.json` можно решать, что делать дальше. Для повторения
строго того же обучения понадобятся также сохранённые версия кода и окружение.


In [14]:
EXPORT_PATH = None
if MODE != "prepare_only" and RUN_DIR is not None:
    from biokg_audit.workflow import export_run
    from IPython.display import FileLink, display
    EXPORT_PATH = export_run(RUN_DIR)
    print("Архив результатов:", EXPORT_PATH)
    display(FileLink(str(EXPORT_PATH)))
else:
    print("Эксперимент не запускался; результатов для выгрузки нет.")


Архив результатов: /kaggle/working/dkr_c3/runs/20261112T122115_1794486075115976605_baselines_6dc8c4bb_export_1794486123133191974.zip


/kaggle/working/dkr_c3/runs/20261112T122115_1794486075115976605_baselines_6dc8c4bb_export_1794486123133191974.zip

## 9. Что делать после первого успешного запуска

1. Проверить audit и таблицу baseline. Не отключать проверки ради красивого числа.
2. На T4 выполнить один короткий pilot, посмотреть время, память и сохранение весов.
3. С тем же budget отдельно сравнить HeadBlind, TransE и ComplEx.
4. После проверки кода определить общий бюджет и повторить обучение с несколькими seeds.
5. Затем реализовать matched-negatives, доверительные интервалы и финальную оценку.

**Сейчас не реализованы:** final-test lock/evaluation, matched protocol, bootstrap/CI,
полный experiment sweep и проверка воспроизведения после cloud restart.
Функция строгого resume есть в training.py, но путь переноса checkpoint между
сессиями этим notebook ещё не проверен. Новый запуск не продолжает старый автоматически.

Не сравнивай pilot напрямую с опубликованным leaderboard: размерность, обучение
и оценочная выборка отличаются. Собственные результаты появятся только после твоего запуска.

Источники: [OGB](https://ogb.stanford.edu/docs/linkprop/#ogbl-biokg),
[официальный evaluator](https://github.com/snap-stanford/ogb/blob/master/ogb/linkproppred/evaluate.py),
[формат split](https://github.com/snap-stanford/ogb/blob/master/ogb/linkproppred/dataset.py),
[NVIDIA T4, 16 GB](https://www.nvidia.com/en-us/data-center/tesla-t4/).
